# Bibliotheque + lecture Excel

In [1]:
import pandas as pd 
import numpy as np 
import os
import re
import datetime
from collections import defaultdict
import zipfile
from datetime import datetime, timedelta
from pathlib import Path

In [2]:
df = pd.read_excel('AGING_Ceinture_Montre (avec groupe).xlsx',sheet_name=None)

C:\Users\judupont\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():
C:\Users\judupont\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():


In [3]:
df.keys()

dict_keys(['APHM', 'CAEN', 'POITIERS', 'ROUEN', 'LAVERAN', 'LPC', 'SAINTE-MARGUERITE', 'CGD', 'CERCA', 'Effectifs ceinture', 'Effectifs montre', 'CODEBOOK '])

# Vérification blocs

In [4]:
def convertir_heure_excel_ou_texte(x):
    if pd.isna(x):
        return pd.NaT
    if isinstance(x, (int, float)):
        # Excel stocke l'heure comme fraction de jour
        return pd.to_datetime(x, unit="D", origin="1899-12-30")
    s = str(x).strip().replace("h", ":")  # gère '13h58'
    try:
        return pd.to_datetime(s, format="%H:%M:%S")  # ajoute secondes
    except Exception:
        try:
            return pd.to_datetime(s, format="%H:%M")     # fallback H:M
        except Exception:
            try:
                return pd.to_datetime(s)                 # Pandas devine le format
            except Exception:
                return pd.NaT


In [5]:
def analyser_feuille_bloc(
    df,
    nom_feuille,
    col_debut,
    col_fin,
    nom_bloc
):
    print(f"\n--- Feuille : {nom_feuille} | {nom_bloc} ---")

    col_id = "Numero_inclusion"

    # ===============================
    # 🔧 DEBUG : normalisation ID
    # ===============================
    df = df.copy()
    df[col_id] = df[col_id].astype(str).str.upper().str.strip()

    # ==========================================================
    # 1️⃣ RÈGLES SPÉCIFIQUES CANDIDATS
    # ==========================================================
    REGLES_CANDIDATS = {
        ("CAEN", "bloc3", "0303PAR"): {"col_debut": "heure_rappel_cond_v2"},

        ("CAEN", "bloc3", "0305LCR"): {"col_debut": "heure_fluence_sem_v2"},

        ("CAEN", "bloc2S", "0314CDS"): {"col_fin": "heure_fluence_sem_v2"},
        ("CAEN", "bloc3", "0314CDS"): {"col_debut": "heure_fluence_sem_v2"},
        
        ("CAEN", "bloc2S", "0335JMS"): {"col_fin": "heure_fluence_sem_v2"},
        ("CAEN", "bloc3", "0335JMS"): {"col_debut": "heure_fluence_sem_v2"},

        ("CAEN", "bloc3", "0339NPR"): {"col_debut": "heure_fluence_sem_v2"},

        ("CAEN", "bloc2S", "0346GLS"): { "col_fin": "heure_fluence_sem_v2"},
        ("CAEN", "bloc3", "0346GLS"): {"col_debut": "heure_fluence_sem_v2"},

        ("CAEN", "bloc2R", "0353LNR"): { "col_fin": "heure_fluence_sem_v2"},
        ("CAEN", "bloc3", "0353LNR"): {"col_debut": "heure_fluence_sem_v2"},

        ("POITIERS", "bloc2S", "0414PJS"): {"col_fin": "heure_fluence_sem_v2"},
        ("POITIERS", "bloc3", "0414PJS"): {"col_debut": "heure_fluence_sem_v2"},

        ("CGD", "bloc2S", "0902SIS"): {"col_fin": "heure_fluence_sem_v2"},
        ("CGD", "bloc3", "0902SIS"): {"col_debut": "heure_fluence_sem_v2"},

        ("CGD", "bloc3", "0905SLR"): {"col_debut": "heure_nback_v2"},

        ("CGD", "bloc3", "0906DNR"): {"col_debut": "heure_fluence_sem_v2"},

        ("CERCA", "bloc3", "0414PVR"): {"col_debut": "heure_nback_v2"},

        ("CERCA", "bloc3", "0419SRR"): {"col_debut": "heure_nback_v2"},

        ("CERCA", "bloc3", "0420MCR"): {"col_debut": "heure_nback_v2"},

        ("CERCA", "bloc2S", "0421ACS"): {"col_fin": "heure_nback_v2"},
        ("CERCA", "bloc3", "0421ACS"): {"col_debut": "heure_nback_v2"},

        ("CERCA", "bloc3", "0423RMR"): {"col_debut": "heure_nback_v2"},

        ("CERCA", "bloc3", "0424BLR"): {"col_debut": "heure_nback_v2"},
    }


    # ==========================================================
    # 2️⃣ INITIALISATION COLONNES
    # ==========================================================
    df["_col_debut"] = col_debut
    df["_col_fin"] = col_fin

    # ==========================================================
    # 3️⃣ RÈGLES GLOBALES
    # ==========================================================
    if nom_feuille == "SAINTE-MARGUERITE" and nom_bloc == "bloc2S":
        df["_col_fin"] = "heure_nback_v2"

    if nom_feuille == "SAINTE-MARGUERITE" and nom_bloc == "bloc3":
        df["_col_debut"] = "heure_nback_v2"

    # ==========================================================
    # 4️⃣ RÈGLES SPÉCIFIQUES (écrasent le global)
    # ==========================================================
    print("\n🔎 Vérification règles spécifiques :")

    for (centre, bloc, candidat), regle in REGLES_CANDIDATS.items():

        if centre != nom_feuille or bloc != nom_bloc:
            continue

        mask = df[col_id] == candidat

        if mask.any():
            print(f"🔧 Règle appliquée pour {candidat}")
            if "col_debut" in regle:
                df.loc[mask, "_col_debut"] = regle["col_debut"]
            if "col_fin" in regle:
                df.loc[mask, "_col_fin"] = regle["col_fin"]
        else:
            print(f"⚠️ {candidat} non trouvé dans cette feuille")

    # ==========================================================
    # 6️⃣ CONVERSION HEURES
    # ==========================================================
    df["_debut_ts"] = df.apply(
        lambda r: convertir_heure_excel_ou_texte(r[r["_col_debut"]]),
        axis=1
    )

    df["_fin_ts"] = df.apply(
        lambda r: convertir_heure_excel_ou_texte(r[r["_col_fin"]]),
        axis=1
    )

    # ==========================================================
    # 7️⃣ CALCUL DURÉE
    # ==========================================================
    col_duree = f"duree_{nom_bloc}"
    col_rejet = f"rejeter_{nom_bloc}"

    df[col_duree] = (df["_fin_ts"] - df["_debut_ts"]).dt.total_seconds() / 60
    df.loc[df[col_duree] < 0, col_duree] += 24 * 60

    # ==========================================================
    # 8️⃣ STATISTIQUES
    # ==========================================================
    mask_manquant = df[col_duree].isna()
    mask_valide = df[col_duree].notna() & (df[col_duree] > 0)

    moyenne = df.loc[mask_valide, col_duree].mean()
    ecart_type = df.loc[mask_valide, col_duree].std()

    print(f"\nMoyenne = {moyenne:.2f} min")
    print(f"Écart-type = {ecart_type:.2f} min")

    borne_inf = moyenne - 2 * ecart_type

    df[col_rejet] = 0
    df.loc[mask_manquant, col_rejet] = 2

    mask_rejet = df[col_duree] < borne_inf

    if nom_bloc == "bloc1":
        mask_rejet = mask_rejet | (df[col_duree] < 10)

    df.loc[mask_valide & mask_rejet, col_rejet] = 1

    # ==========================================================
    # 9️⃣ DEBUG CANDIDATS PROBLÉMATIQUES
    # ==========================================================
    print("\n🔎 DEBUG Candidats spécifiques :")

    candidats_test = [r[2] for r in REGLES_CANDIDATS.keys()]
    candidats_test = list(set(candidats_test))

    print(
        df.loc[
            df[col_id].isin(candidats_test),
            [col_id, col_duree, col_rejet]
        ]
    )

    # ==========================================================
    # 10️⃣ NETTOYAGE
    # ==========================================================
    df = df.drop(columns=["_col_debut", "_col_fin", "_debut_ts", "_fin_ts"])

    print(f"\nRésumé {col_rejet} :")
    print(df[col_rejet].value_counts().sort_index())

    return {
        "df": df,
        "moyenne": moyenne,
        "ecart_type": ecart_type,
        "rejets": {
            "manquant": df.loc[df[col_rejet] == 2, col_id].tolist(),
            "rejetes": df.loc[df[col_rejet] == 1, col_id].tolist()
        }
    }

## Bloc 1

In [6]:
feuilles = [
    'APHM', 'CAEN', 'POITIERS', 'ROUEN',
    'LAVERAN', 'LPC', 'SAINTE-MARGUERITE',
    'CGD', 'CERCA'
]

resultats_bloc1 = {}

for feuille in feuilles:
    print("\n" + "=" * 60)

    resultats_bloc1[feuille] = analyser_feuille_bloc(
        df=df[feuille],
        nom_feuille=feuille,
        col_debut="heure_montre_v2",   
        col_fin="heure_fin_anamnese_v2",        
        nom_bloc="bloc1"
    )




--- Feuille : APHM | bloc1 ---

🔎 Vérification règles spécifiques :

Moyenne = 24.92 min
Écart-type = 10.54 min

🔎 DEBUG Candidats spécifiques :
Empty DataFrame
Columns: [Numero_inclusion, duree_bloc1, rejeter_bloc1]
Index: []

Résumé rejeter_bloc1 :
rejeter_bloc1
0    25
Name: count, dtype: int64


--- Feuille : CAEN | bloc1 ---

🔎 Vérification règles spécifiques :

Moyenne = 19.05 min
Écart-type = 7.06 min

🔎 DEBUG Candidats spécifiques :
   Numero_inclusion  duree_bloc1  rejeter_bloc1
2           0303PAR         23.0              0
4           0305LCR         14.0              0
16          0314CDS         20.0              0
34          0335JMS         20.0              0
38          0339NPR         23.0              0
47          0346GLS         26.0              0
51          0353LNR         20.0              0

Résumé rejeter_bloc1 :
rejeter_bloc1
0    54
1     2
Name: count, dtype: int64


--- Feuille : POITIERS | bloc1 ---

🔎 Vérification règles spécifiques :

Moyenne = 30.2

## Bloc 2 

### Avec vidéo: de heure_rlri16imm_debut_V2 à heure_rappel_cond_v2

In [7]:
resultats_bloc2_R = {}

for feuille in feuilles:
    print("\n" + "=" * 60)

    resultats_bloc2_R[feuille] = analyser_feuille_bloc(
        df=df[feuille],
        nom_feuille=feuille,
        col_debut="heure_rlri16imm_debut_V2",  
        col_fin= "heure_rappel_cond_v2" ,  # Beaucoup de rejet car tous les S ont des NV car pas vu la vidéo         
        nom_bloc="bloc2R"
    )




--- Feuille : APHM | bloc2R ---

🔎 Vérification règles spécifiques :

Moyenne = 44.92 min
Écart-type = 7.48 min

🔎 DEBUG Candidats spécifiques :
Empty DataFrame
Columns: [Numero_inclusion, duree_bloc2R, rejeter_bloc2R]
Index: []

Résumé rejeter_bloc2R :
rejeter_bloc2R
0    13
2    12
Name: count, dtype: int64


--- Feuille : CAEN | bloc2R ---

🔎 Vérification règles spécifiques :
🔧 Règle appliquée pour 0353LNR

Moyenne = 39.39 min
Écart-type = 5.17 min

🔎 DEBUG Candidats spécifiques :
   Numero_inclusion  duree_bloc2R  rejeter_bloc2R
2           0303PAR          43.0               0
4           0305LCR          44.0               0
16          0314CDS           NaN               2
34          0335JMS           NaN               2
38          0339NPR          34.0               0
47          0346GLS           NaN               2
51          0353LNR          39.0               0

Résumé rejeter_bloc2R :
rejeter_bloc2R
0    27
1     1
2    28
Name: count, dtype: int64


--- Feuille : POI

resultats_bloc2_R

### Sans vidéo: de heure_rlri16imm_debut_V2 à heure_nback_v2

In [8]:
resultats_bloc2_S = {}

for feuille in feuilles:
    print("\n" + "=" * 60)

    resultats_bloc2_S[feuille] = analyser_feuille_bloc(
        df=df[feuille],
        nom_feuille=feuille,
        col_debut="heure_rlri16imm_debut_V2",
        col_fin="heure_nback_v2 (consigne)",
        nom_bloc="bloc2S"
    )



--- Feuille : APHM | bloc2S ---

🔎 Vérification règles spécifiques :

Moyenne = 46.44 min
Écart-type = 9.26 min

🔎 DEBUG Candidats spécifiques :
Empty DataFrame
Columns: [Numero_inclusion, duree_bloc2S, rejeter_bloc2S]
Index: []

Résumé rejeter_bloc2S :
rejeter_bloc2S
0    25
Name: count, dtype: int64


--- Feuille : CAEN | bloc2S ---

🔎 Vérification règles spécifiques :
🔧 Règle appliquée pour 0314CDS
🔧 Règle appliquée pour 0335JMS
🔧 Règle appliquée pour 0346GLS

Moyenne = 40.08 min
Écart-type = 7.35 min

🔎 DEBUG Candidats spécifiques :
   Numero_inclusion  duree_bloc2S  rejeter_bloc2S
2           0303PAR           NaN               2
4           0305LCR           NaN               2
16          0314CDS          54.0               0
34          0335JMS          53.0               0
38          0339NPR           NaN               2
47          0346GLS          40.0               0
51          0353LNR           NaN               2

Résumé rejeter_bloc2S :
rejeter_bloc2S
0    49
1     2

## Bloc 3 

In [9]:
resultats_bloc3 = {}

for feuille in feuilles:
    print("\n" + "=" * 60)

    resultats_bloc3[feuille] = analyser_feuille_bloc(
        df=df[feuille],
        nom_feuille=feuille,
        col_debut="heure_nback_v2 (consigne)", 
        col_fin="heure_fin_tests_v2",        
        nom_bloc="bloc3"
    )



--- Feuille : APHM | bloc3 ---

🔎 Vérification règles spécifiques :

Moyenne = 48.84 min
Écart-type = 8.85 min

🔎 DEBUG Candidats spécifiques :
Empty DataFrame
Columns: [Numero_inclusion, duree_bloc3, rejeter_bloc3]
Index: []

Résumé rejeter_bloc3 :
rejeter_bloc3
0    24
1     1
Name: count, dtype: int64


--- Feuille : CAEN | bloc3 ---

🔎 Vérification règles spécifiques :
🔧 Règle appliquée pour 0303PAR
🔧 Règle appliquée pour 0305LCR
🔧 Règle appliquée pour 0314CDS
🔧 Règle appliquée pour 0335JMS
🔧 Règle appliquée pour 0339NPR
🔧 Règle appliquée pour 0346GLS
🔧 Règle appliquée pour 0353LNR

Moyenne = 43.02 min
Écart-type = 10.68 min

🔎 DEBUG Candidats spécifiques :
   Numero_inclusion  duree_bloc3  rejeter_bloc3
2           0303PAR         22.0              0
4           0305LCR         40.0              0
16          0314CDS         19.0              1
34          0335JMS         18.0              1
38          0339NPR         29.0              0
47          0346GLS         25.0        

resultats_bloc3

# Nombre de candidats qui passent toutes les conditions 

In [10]:
'''
Un candidat est conservé si nous pouvons calculer la durée de tous ses blocs et si celles-ci respectent les conditions suivantes :
    - bloc 1 > 10 min
    - bloc2 et bloc 3 > mean - 2σ min
'''
    
def candidats_valides_tous_blocs_depuis_resultats(
    feuille,
    resultats_blocs,
    col_id="Numero_inclusion"
):
    """
    resultats_blocs = dict {
        "bloc1": resultats_bloc1,
        "bloc2S": resultats_bloc2_S,
        "bloc2R": resultats_bloc2_R,
        "bloc3": resultats_bloc3
    }
    """

    # Récupération du df de référence (bloc1 par ex)
    df_ref = resultats_blocs["bloc1"][feuille]["df"].copy()

    colonnes_rejet = []

    # Ajouter chaque colonne de rejet depuis chaque bloc
    for nom_bloc, res_bloc in resultats_blocs.items():
        col_rejet = f"rejeter_{nom_bloc}"
        if col_rejet not in res_bloc[feuille]["df"].columns:
            print(f"Colonne manquante : {col_rejet} dans {feuille}")
            return None

        df_ref[col_rejet] = res_bloc[feuille]["df"][col_rejet]
        colonnes_rejet.append(col_rejet)

    # Condition : tout à 0
    mask_valide = (df_ref[colonnes_rejet] == 0).all(axis=1)

    candidats_ok = df_ref.loc[mask_valide, col_id].tolist()
    candidats_rejetes = df_ref.loc[~mask_valide, col_id].tolist()

    print(f"\n=== {feuille} ===")
    print(f"Candidats valides sur TOUS les blocs : {len(candidats_ok)}")

    print("Liste des candidats conservés :")
    for pid in candidats_ok:
        print(f" - {pid}")

    return {
        "nb_valides": len(candidats_ok),
        "valides": candidats_ok,
        "rejetes": candidats_rejetes,
    }


In [11]:
resultats_globaux = {}

colonnes_blocs = [
    "rejeter_bloc1",
    "rejeter_bloc2S",
    "rejeter_bloc2R",
    "rejeter_bloc3"
]

for feuille in feuilles:
    print("\n" + "=" * 70)
    print(f"Analyse globale – Feuille : {feuille}")

    # ===== Base : bloc 1 =====
    df_global = resultats_bloc1[feuille]["df"][
        ["Numero_inclusion", "Condition", "rejeter_bloc1"]
    ].copy()

    # ===== Fusion bloc 2S =====
    df_global = df_global.merge(
        resultats_bloc2_S[feuille]["df"][
            ["Numero_inclusion", "rejeter_bloc2S"]
        ],
        on="Numero_inclusion",
        how="left"
    )

    # ===== Fusion bloc 2R =====
    df_global = df_global.merge(
        resultats_bloc2_R[feuille]["df"][
            ["Numero_inclusion", "rejeter_bloc2R"]
        ],
        on="Numero_inclusion",
        how="left"
    )

    # ===== Fusion bloc 3 =====
    df_global = df_global.merge(
        resultats_bloc3[feuille]["df"][
            ["Numero_inclusion", "rejeter_bloc3"]
        ],
        on="Numero_inclusion",
        how="left"
    )

    # ===== Affichage debug AVANT filtre =====
    print("\n📊 df_global AVANT fillna et filtres :")
    display(df_global)

    # ===== Sécurité : NaN → rejet (2) =====
    df_global[colonnes_blocs] = df_global[colonnes_blocs].fillna(2)

    # ===== Filtrage conditionnel S / R =====
    mask_S = (
        (df_global["Condition"] == "Standard") &
        (df_global["rejeter_bloc1"] == 0) &
        (df_global["rejeter_bloc2S"] == 0) &
        (df_global["rejeter_bloc3"] == 0)
    )

    mask_R = (
        (df_global["Condition"] == "Réduction de la menace") &
        (df_global["rejeter_bloc1"] == 0) &
        (df_global["rejeter_bloc2R"] == 0) &
        (df_global["rejeter_bloc3"] == 0)
    )

    mask_valide = mask_S | mask_R

    # ===== Extraction =====
    candidats_valides = df_global.loc[
        mask_valide, "Numero_inclusion"
    ].tolist()

    candidats_rejetes = df_global.loc[
        ~mask_valide, "Numero_inclusion"
    ].tolist()

    # ===== Résumés =====
    print(f"\nNombre total de candidats : {len(df_global)}")
    print(f"Candidats VALIDES (selon Condition) : {len(candidats_valides)}")
    print(f"Candidats REJETÉS : {len(candidats_rejetes)}")

    print("\nCandidats valides par condition :")
    print(df_global.loc[mask_valide, "Condition"].value_counts())

    print("\nListe des candidats valides :")
    for pid in candidats_valides:
        print(f" - {pid}")

    # ===== Stockage =====
    resultats_globaux[feuille] = {
        "df": df_global,
        "valides": candidats_valides,
        "rejetes": candidats_rejetes,
        "n_valides": len(candidats_valides),
        "n_rejetes": len(candidats_rejetes),
    }



Analyse globale – Feuille : APHM

📊 df_global AVANT fillna et filtres :


,Numero_inclusion,Condition,rejeter_bloc1,rejeter_bloc2S,rejeter_bloc2R,rejeter_bloc3
0,0101CAR,Réduction de la menace,0,0,0,0
1,0102PCR,Réduction de la menace,0,0,0,0
2,0103SHS,Standard,0,0,2,0
3,0105PNR,Réduction de la menace,0,0,0,0
4,0104FJS,Standard,0,0,2,0
5,0106JLS,Standard,0,0,2,0
6,0107DSS,Standard,0,0,2,0
7,0108BFS,Standard,0,0,2,0
8,0109GSS,Standard,0,0,2,0
9,0110LPR,Réduction de la menace,0,0,0,0



Nombre total de candidats : 25
Candidats VALIDES (selon Condition) : 24
Candidats REJETÉS : 1

Candidats valides par condition :
Condition
Réduction de la menace    13
Standard                  11
Name: count, dtype: int64

Liste des candidats valides :
 - 0101CAR
 - 0102PCR
 - 0103SHS
 - 0105PNR
 - 0104FJS
 - 0106JLS
 - 0107DSS
 - 0108BFS
 - 0109GSS
 - 0110LPR
 - 0111MNR
 - 0112BSR
 - 0113BMR
 - 0114LMS
 - 0115CJR
 - 0116VAR
 - 0117POS
 - 0118TNS
 - 0119LJS
 - 0120LCR
 - 0121MJR
 - 0123RMS
 - 0124BAR
 - 0125BMR

Analyse globale – Feuille : CAEN

📊 df_global AVANT fillna et filtres :


,Numero_inclusion,Condition,rejeter_bloc1,rejeter_bloc2S,rejeter_bloc2R,rejeter_bloc3
0,0301GNR,Réduction de la menace,0,0,0,0
1,0302ZMR,Réduction de la menace,0,0,0,0
2,0303PAR,Réduction de la menace,0,2,0,0
3,0304VMR,Réduction de la menace,0,0,0,0
4,0305LCR,Réduction de la menace,0,2,0,0
5,0307SMR,Réduction de la menace,0,0,0,0
6,0306MDR,Réduction de la menace,0,0,0,0
7,0308RGR,Réduction de la menace,0,0,0,0
8,0309DBS,Standard,0,0,2,0
9,0311CJS,Standard,0,0,2,0



Nombre total de candidats : 56
Candidats VALIDES (selon Condition) : 50
Candidats REJETÉS : 6

Candidats valides par condition :
Condition
Réduction de la menace    26
Standard                  24
Name: count, dtype: int64

Liste des candidats valides :
 - 0301GNR
 - 0302ZMR
 - 0303PAR
 - 0304VMR
 - 0305LCR
 - 0307SMR
 - 0306MDR
 - 0308RGR
 - 0309DBS
 - 0311CJS
 - 0310ACS
 - 0315VCS
 - 0318PMS
 - 0313FPR
 - 0312JMS
 - 0316TMS
 - 0317LGS
 - 0319LJS
 - 0320DGS
 - 0321DRR
 - 0323RFS
 - 0322RMS
 - 0324LMR
 - 0325LCR
 - 0327IAR
 - 0326BJR
 - 0328MPR
 - 0330RTR
 - 0329NMR
 - 0331GRS
 - 0332MCR
 - 0333MMS
 - 0334TVS
 - 0336CBS
 - 0337BFS
 - 0339NPR
 - 0341LJS
 - 0342VNS
 - 0343PAS
 - 0345LAR
 - 0349LLS
 - 0347DMR
 - 0346GLS
 - 0348GCR
 - 0350MYR
 - 0351FIR
 - 0352RNR
 - 0354GKR
 - 0355BAS
 - 0356DMS

Analyse globale – Feuille : POITIERS

📊 df_global AVANT fillna et filtres :


...


Nombre total de candidats : 8
Candidats VALIDES (selon Condition) : 6
Candidats REJETÉS : 2

Candidats valides par condition :
Condition
Réduction de la menace    4
Standard                  2
Name: count, dtype: int64

Liste des candidats valides :
 - 0801HDR
 - 0802LAS
 - 0804GOR
 - 0805BMS
 - 0807OMR
 - 0808PJR

Analyse globale – Feuille : CGD

📊 df_global AVANT fillna et filtres :


,Numero_inclusion,Condition,rejeter_bloc1,rejeter_bloc2S,rejeter_bloc2R,rejeter_bloc3
0,0901SMR,Réduction de la menace,0,0,0,0
1,0903DJS,Standard,2,0,2,0
2,0902SIS,Standard,0,0,2,0
3,0906DNR,Réduction de la menace,0,2,0,0
4,0904KJS,Standard,0,0,2,0
5,0905SLR,Réduction de la menace,0,2,0,0



Nombre total de candidats : 6
Candidats VALIDES (selon Condition) : 5
Candidats REJETÉS : 1

Candidats valides par condition :
Condition
Réduction de la menace    3
Standard                  2
Name: count, dtype: int64

Liste des candidats valides :
 - 0901SMR
 - 0902SIS
 - 0906DNR
 - 0904KJS
 - 0905SLR

Analyse globale – Feuille : CERCA

📊 df_global AVANT fillna et filtres :


,Numero_inclusion,Condition,rejeter_bloc1,rejeter_bloc2S,rejeter_bloc2R,rejeter_bloc3
0,0401BDS,Standard,2,2,2,2
1,0402TFR,Réduction de la menace,2,2,2,2
2,0404DER,Réduction de la menace,2,2,2,2
3,0405RCR,Réduction de la menace,2,2,2,2
4,0406BBS,Standard,2,2,2,2
5,0407HMS,Standard,2,2,2,2
6,0408SJS,Standard,2,2,2,2
7,0409HCR,Réduction de la menace,2,2,2,2
8,0410FMS,Standard,2,2,2,2
9,0411NPR,Réduction de la menace,2,2,2,2



Nombre total de candidats : 40
Candidats VALIDES (selon Condition) : 14
Candidats REJETÉS : 26

Candidats valides par condition :
Condition
Réduction de la menace    9
Standard                  5
Name: count, dtype: int64

Liste des candidats valides :
 - 0413MMR
 - 0414PVR
 - 0415HJS
 - 0417FAS
 - 0417KMS
 - 0418MPR
 - 0418TJR
 - 0419SRR
 - 0420MCR
 - 0420LMR
 - 0421ACS
 - 0422WCS
 - 0423RMR
 - 0424BLR


In [12]:
total_valides = sum(
    res["n_valides"] for res in resultats_globaux.values()
)

print("=" * 70)
print(f"✅ Nombre TOTAL de candidats valides (toutes feuilles confondues) : {total_valides}")

✅ Nombre TOTAL de candidats valides (toutes feuilles confondues) : 169


# 169 candidats retenus

# Recherche des candidats et téléchargement de leur fichier EDA V2

In [13]:
import pandas as pd

# Afficher toutes les lignes
pd.set_option('display.max_rows', None)

# Afficher toutes les colonnes
pd.set_option('display.max_columns', None)

# Afficher toute la largeur de chaque colonne
pd.set_option('display.max_colwidth', None)

## APHM

In [14]:
# ===== Racine des patients =====
racine_aphm = r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran"

# ===== Candidats valides =====
candidats_aphm = [c.upper() for c in resultats_globaux["APHM"]["valides"]]

# ===== Mapping PID → Condition =====
df_conditions = resultats_globaux["APHM"]["df"][["Numero_inclusion", "Condition"]].copy()
df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()
df_conditions["Condition"] = df_conditions["Condition"].astype(str)
dict_condition = dict(zip(df_conditions["Numero_inclusion"], df_conditions["Condition"]))

rows = []

for root, dirs, files in os.walk(racine_aphm):

    if not any("V2" in d.upper() for d in root.split(os.sep)):
        continue

    for f in files:
        if not f.lower().endswith(".csv"):
            continue
        if "EDA" not in f.upper():  
            continue

        chemin = os.path.join(root, f)

        # ===== Numero_inclusion = dossier juste après la racine =====
        rel_path = os.path.relpath(chemin, racine_aphm)
        parts = rel_path.split(os.sep)
        pid_trouve = parts[0].upper()

        if pid_trouve not in candidats_aphm:
            continue

        # ===== Timestamp Unix (dossier juste avant EDA.csv) =====
        folder = os.path.basename(os.path.dirname(chemin))
        timestamp_match = re.match(r"(\d+)_", folder)
        timestamp_unix = int(timestamp_match.group(1)) if timestamp_match else None

        # ===== Conversion Unix → date locale =====
        date_eda = datetime.datetime.fromtimestamp(timestamp_unix) if timestamp_unix else None

        # ===== Condition =====
        condition = dict_condition.get(pid_trouve)
        if condition is None:
            print(f"⚠️ Condition manquante pour {pid_trouve} → ignoré")
            continue

        rows.append({
            "Numero_inclusion": pid_trouve,
            "Chemin": chemin,
            "Condition": condition,
            "Feuille": "APHM",
            "Timestamp_unix": timestamp_unix,
            "Date_eda": date_eda
        })

# ===== DataFrame complet =====
df_aphm = pd.DataFrame(rows, columns=["Numero_inclusion", "Chemin", "Condition", "Feuille", "Timestamp_unix", "Date_eda"])

# ===== Suppression des doublons =====
if not df_aphm.empty:
    df_aphm_final = (
        df_aphm
        .assign(priorite_montre=df_aphm["Chemin"].str.contains("MONTRE", case=False))
        .sort_values(["Numero_inclusion", "priorite_montre"], ascending=[True, False])
        .drop_duplicates(subset="Numero_inclusion", keep="first")
        .drop(columns="priorite_montre")
        .reset_index(drop=True)
    )
else:
    df_aphm_final = df_aphm.copy()

print(f"Candidats attendus APHM : {len(candidats_aphm)}")
print(f"Fichiers EDA APHM V2 retenus : {len(df_aphm_final)}")

display(df_aphm_final)


Candidats attendus APHM : 24
Fichiers EDA APHM V2 retenus : 24


,Numero_inclusion,Chemin,Condition,Feuille,Timestamp_unix,Date_eda
0,0101CAR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0101CAR\0101CAR 06-07-2018 V2\Montre 0101CAR\1530858300_A01093\EDA.csv,Réduction de la menace,APHM,1530858300,2018-07-06 08:25:00
1,0102PCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0102PCR\0102PCR 09-07-2018 V2\Montre 0102PCR\1531119483_A01093\EDA.csv,Réduction de la menace,APHM,1531119483,2018-07-09 08:58:03
2,0103SHS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0103SHS\0103SHS 10-09-2018 V2\Montre 0103SHS\1536563089_A01093\EDA.csv,Standard,APHM,1536563089,2018-09-10 09:04:49
3,0104FJS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0104FJS\0104FJS 29-11-2018 V2\Montre 0104FJS\1543476632_A01093\EDA.csv,Standard,APHM,1543476632,2018-11-29 08:30:32
4,0105PNR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0105PNR\0105PNR 10-12-2018 V2\Montre 0105PNR\1544426613_A01093\EDA.csv,Réduction de la menace,APHM,1544426613,2018-12-10 08:23:33
5,0106JLS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0106JLS\0106JLS 03-01-2019 V2\Montre 0106JLS\1546500427_A01093\EDA.csv,Standard,APHM,1546500427,2019-01-03 08:27:07
6,0107DSS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0107DSS\0107DSS 14-01-2019 V2\Montre 0107DSS\1547450742_A01093\EDA.csv,Standard,APHM,1547450742,2019-01-14 08:25:42
7,0108BFS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0108BFS\0108BFS 17-01-2019 V2\Montre 0108BFS\1547711087_A01093\EDA.csv,Standard,APHM,1547711087,2019-01-17 08:44:47
8,0109GSS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0109GSS\0109GSS 18-02-2019 V2\Montre 0109GSS\1550476660_A01093\EDA.csv,Standard,APHM,1550476660,2019-02-18 08:57:40
9,0110LPR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0110LPR\0110LPR 26-02-2019 V2\Montre 0110LPR\1551167333_A01093\EDA.csv,Réduction de la menace,APHM,1551167333,2019-02-26 08:48:53


## CAEN

In [15]:
# ===== Racine et candidats CAEN =====
racine_caen = r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN"
candidats_caen = [c.upper().replace(" ", "") for c in resultats_globaux["CAEN"]["valides"]]

# ===== DataFrame Excel =====
df_excel_caen = df['CAEN'][["Numero_inclusion", "Condition", "Date_v2"]].copy()
df_excel_caen["Numero_inclusion"] = df_excel_caen["Numero_inclusion"].str.upper().str.replace(" ", "")
df_excel_caen["Condition"] = df_excel_caen["Condition"].astype(str)
df_excel_caen["Date_v2"] = pd.to_datetime(df_excel_caen["Date_v2"], dayfirst=True).dt.date

# ===== Dictionnaires pour lookup rapide =====
dict_condition = dict(zip(df_excel_caen["Numero_inclusion"], df_excel_caen["Condition"]))
dict_date_v2 = dict(zip(df_excel_caen["Numero_inclusion"], df_excel_caen["Date_v2"]))

# ===== Paths forcés pour candidats spécifiques =====
paths_forces = {
    "0306MDR": r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1557145160_A012950306 MDR\EDA.csv",
    "0356DMS": r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1663832640_A012950356DMS\EDA.csv",
    "0305LCR":r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1556179294_A01295 0305 LCR\EDA.csv"
}
paths_forces = {k: os.path.normpath(v).upper() for k, v in paths_forces.items()}

# ===== Pattern PID dans le chemin (hors paths forcés) =====
patterns_candidats = {
    pid: re.compile(rf"\b{pid}\b", re.IGNORECASE)
    for pid in candidats_caen if pid not in paths_forces
}

rows = []

# ===== Parcours des fichiers =====
for root, dirs, files in os.walk(racine_caen):
    for f in files:
        if not f.lower().endswith(".csv"):
            continue
        if "EDA" not in f.upper():
            continue

        chemin = os.path.join(root, f)
        chemin_norm = os.path.normpath(chemin).upper()

        # ===== Vérification regex pour candidats normaux =====
        pid_trouve = None
        for pid, pat in patterns_candidats.items():
            if pat.search(chemin):
                pid_trouve = pid
                break

        # ===== Vérification chemins forcés =====
        if chemin_norm in paths_forces.values():
            pid_trouve = [k for k, v in paths_forces.items() if v == chemin_norm][0]

        if not pid_trouve:
            continue

        # ===== Timestamp Unix =====
        folder = os.path.basename(os.path.dirname(chemin))
        match = re.match(r"(\d+)_A\d+\s*(.+)", folder)
        timestamp_unix = int(match.group(1)) if match else None
        date_eda = datetime.datetime.fromtimestamp(timestamp_unix).date() if timestamp_unix else None

        # ===== Condition et date Excel =====
        condition = dict_condition.get(pid_trouve)
        date_v2_excel = dict_date_v2.get(pid_trouve)

        # ===== Filtre date sauf pour chemins forcés =====
        if chemin_norm not in paths_forces.values() and date_eda and date_v2_excel:
            if date_eda > date_v2_excel:
                continue

        rows.append({
            "Numero_inclusion": pid_trouve,
            "Chemin": chemin,
            "Condition": condition,
            "Feuille": "CAEN",
            "Timestamp_unix": timestamp_unix,
            "Date_eda": date_eda,
            "Date_v2_excel": date_v2_excel
        })

# ===== DataFrame complet =====
df_caen = pd.DataFrame(rows)

# ===== Suppression des doublons =====
if not df_caen.empty:
    df_caen_final = df_caen.groupby("Numero_inclusion", as_index=False)\
                           .apply(lambda x: x.loc[x['Date_eda'].idxmin()])\
                           .reset_index(drop=True)
else:
    df_caen_final = df_caen.copy()

print(f"Candidats attendus CAEN : {len(candidats_caen)}")
print(f"Fichiers EDA CAEN V2 retenus : {len(df_caen_final)}")
display(df_caen_final)

# ===== Affichage des candidats manquants =====
candidats_trouves = set(df_caen_final["Numero_inclusion"].tolist())
candidats_manquants = sorted(set(candidats_caen) - candidats_trouves)

if candidats_manquants:
    print("\n⚠️ Candidats manquants dans l'extraction :")
    for c in candidats_manquants:
        print(f" - {c}")
else:
    print("\n✅ Tous les candidats ont été trouvés")


Candidats attendus CAEN : 50
Fichiers EDA CAEN V2 retenus : 49


C:\Users\judupont\AppData\Local\Temp\ipykernel_50772\2733414304.py:93: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.loc[x['Date_eda'].idxmin()])\


,Numero_inclusion,Chemin,Condition,Feuille,Timestamp_unix,Date_eda,Date_v2_excel
0,0301GNR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\fichiers ancien serveur\0301GNR\Montre\1547630037_A01295 0301GNR\EDA.csv,Réduction de la menace,CAEN,1547630037,2019-01-16,2019-01-16
1,0302ZMR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\fichiers ancien serveur\0302ZMR\montre\1548318959_A01295 0302 ZMR\EDA.csv,Réduction de la menace,CAEN,1548318959,2019-01-24,2019-01-24
2,0303PAR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\fichiers ancien serveur\0303PAR\Montre\1552553872_A01295 0303 PAR\EDA.csv,Réduction de la menace,CAEN,1552553872,2019-03-14,2019-03-14
3,0304VMR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\fichiers ancien serveur\0304VMR\Montre\1553502969_A01295 0304 VMR\EDA.csv,Réduction de la menace,CAEN,1553502969,2019-03-25,2019-03-25
4,0305LCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1556179294_A01295 0305 LCR\EDA.csv,Réduction de la menace,CAEN,1556179294,2019-04-25,2019-04-25
5,0306MDR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1557145160_A012950306 MDR\EDA.csv,Réduction de la menace,CAEN,1557145160,2019-05-06,2019-05-06
6,0307SMR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1557735741_A01295 0307SMR\EDA.csv,Réduction de la menace,CAEN,1557735741,2019-05-13,2019-05-13
7,0308RGR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1561452848_A01295 0308RGR\EDA.csv,Réduction de la menace,CAEN,1561452848,2019-06-25,2019-06-25
8,0309DBS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1561634352_A01295 0309DBS\EDA.csv,Standard,CAEN,1561634352,2019-06-27,2019-06-27
9,0310ACS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1562140652_A01295 0310ACS\EDA.csv,Standard,CAEN,1562140652,2019-07-03,2019-07-03



⚠️ Candidats manquants dans l'extraction :
 - 0319LJS


## POITIERS

In [16]:
# ===== Racine POITIERS =====
racine_poitiers = r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY"

# ===== Candidats valides =====
candidats_poitiers = [
    c.upper().replace(" ", "")
    for c in resultats_globaux["POITIERS"]["valides"]
]

# ===== Dictionnaire Numero_inclusion → Condition =====
df_conditions = resultats_globaux["POITIERS"]["df"][["Numero_inclusion", "Condition"]].copy()
df_conditions["Numero_inclusion"] = (
    df_conditions["Numero_inclusion"]
    .str.upper()
    .str.replace(" ", "")
)

dict_condition = dict(
    zip(df_conditions["Numero_inclusion"], df_conditions["Condition"])
)

# ===== Regex POITIERS  =====
# ex : 0401TSS → match "0401 TS"
patterns_candidats = {
    pid: re.compile(
        rf"{pid[:4]}\s*-?\s*{pid[4:6]}",
        re.IGNORECASE
    )
    for pid in candidats_poitiers
}

fichiers_par_candidat = defaultdict(list)

for root, dirs, files in os.walk(racine_poitiers):

    if "V2" not in root.upper():
        continue

    for f in files:
        if f.upper() != "EDA.CSV":
            continue

        chemin = os.path.join(root, f)

        for pid, pat in patterns_candidats.items():
            if pat.search(chemin):
                fichiers_par_candidat[pid].append(chemin)
                break

# ===== Sélection finale + DataFrame =====
rows = []

for pid, fichiers in fichiers_par_candidat.items():

    fichiers = sorted(set(fichiers))  # sécurité doublons exacts

    # priorité MONTRE (logique équivalente à CEINTURE)
    montre = [f for f in fichiers if "montre" in f.lower()]
    chemin_final = montre[0] if montre else fichiers[0]

    rows.append({
        "Numero_inclusion": pid,
        "Chemin": chemin_final,
        "Condition": dict_condition.get(pid),
        "Feuille": "POITIERS"
    })

df_poitiers_final = pd.DataFrame(rows)

# ===== Affichage =====
print(f"Candidats attendus POITIERS : {len(candidats_poitiers)}")
print(f"Fichiers EDA POITIERS V2 retenus : {len(df_poitiers_final)}")

manquants = sorted(
    set(candidats_poitiers) - set(df_poitiers_final["Numero_inclusion"])
)

if manquants:
    print("⚠️ Candidats manquants :", manquants)
else:
    print("✅ Tous les candidats POITIERS sont présents")

display(df_poitiers_final)


Candidats attendus POITIERS : 14
Fichiers EDA POITIERS V2 retenus : 13
⚠️ Candidats manquants : ['0409PHR']


,Numero_inclusion,Chemin,Condition,Feuille
0,0401TSS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0401 TS\0401TSS - V2\0401TSS- montre\EDA.csv,Standard,POITIERS
1,0402LLS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0402 LL\0402LLS- V2\0402LLS- montre\EDA.csv,Standard,POITIERS
2,0403DCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0403 DC\0403DCR-V2\0403DCR- montre\EDA.csv,Réduction de la menace,POITIERS
3,0404CYS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0404 CY\0404CYS-V2\0404YC-montre\EDA.csv,Standard,POITIERS
4,0405FCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0405 FC\0405 FCR-V2\0405FCR- montre\EDA.csv,Réduction de la menace,POITIERS
5,0406MCS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0406 MC\0406 MCS-V2\0406MCS-montre\EDA.csv,Standard,POITIERS
6,0407LJR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0407 LJ\0407LJR-V2\0407LJ-montre\EDA.csv,Réduction de la menace,POITIERS
7,0408BCS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0408 BC\0408 BC V2\0408BC-montre\EDA.csv,Standard,POITIERS
8,0410PGS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0410 PG\0410 PGS - V2\montre\0410PGS\EDA.csv,Standard,POITIERS
9,0411VES,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0411 VE\0411 VES-V2\0411VE montre\EDA.csv,Standard,POITIERS


## ROUEN

In [17]:
# ===== Racine ROUEN =====
racine_rouen = r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER"

# ===== Candidats valides =====
candidats_rouen = [c.upper() for c in resultats_globaux["ROUEN"]["valides"]]

# ===== Dictionnaire Numero_inclusion → Condition =====
df_conditions = resultats_globaux["ROUEN"]["df"][["Numero_inclusion", "Condition"]].copy()
df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()
dict_condition = dict(zip(df_conditions["Numero_inclusion"], df_conditions["Condition"]))

# ===== Patterns candidats : 05-09, 05-11, etc =====
patterns_candidats = {
    pid: re.compile(rf"\b{pid[:2]}-{pid[2:4]}\b", re.IGNORECASE)
    for pid in candidats_rouen
}

rows = []

# ===== Parcours fichiers =====
for root, dirs, files in os.walk(racine_rouen):

    if "V2" not in root.upper():
        continue

    for f in files:

        if not f.lower().endswith(".csv"):
            continue

        if "EDA" not in f.upper():
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        pid_trouve = None
        for pid, pat in patterns_candidats.items():
            if pat.search(chemin_upper):
                pid_trouve = pid
                break

        if not pid_trouve:
            continue

        rows.append({
            "Numero_inclusion": pid_trouve,
            "Chemin": chemin,
            "Condition": dict_condition.get(pid_trouve),
            "Feuille": "ROUEN"
        })

# ===== DataFrame final =====
df_rouen_final = pd.DataFrame(rows)

# ===== Suppression des doublons =====
df_rouen_final = (
    df_rouen_final
    .sort_values("Chemin")
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .reset_index(drop=True)
)

# ===== Affichage =====
print(f"Candidats attendus ROUEN : {len(candidats_rouen)}")
print(f"Fichiers EDA ROUEN V2 retenus : {len(df_rouen_final)}")

manquants = sorted(set(candidats_rouen) - set(df_rouen_final["Numero_inclusion"]))
if manquants:
    print("⚠️ Candidats ROUEN manquants :", manquants)
else:
    print("✅ Tous les candidats ROUEN sont présents")

display(df_rouen_final)


Candidats attendus ROUEN : 14
Fichiers EDA ROUEN V2 retenus : 14
✅ Tous les candidats ROUEN sont présents


,Numero_inclusion,Chemin,Condition,Feuille
0,0501MBS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-01-MB-S\V2\montre 05-01MBS V2\EDA.csv,Standard,ROUEN
1,0502LIR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-02-LI-R\V2\MONTRE\EDA.csv,Réduction de la menace,ROUEN
2,0503LCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-03-LC-R\V2\MONTRE\EDA.csv,Réduction de la menace,ROUEN
3,0504BCS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-04-BC-S\V2\montre 05-04-BC -S V2\1556006396_A012E9\EDA.csv,Standard,ROUEN
4,0505PPR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-05-PP-R\V2\MONTRE\EDA.csv,Réduction de la menace,ROUEN
5,0506VMS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-06-VM-S\V2\MONTRE05-06-VM-S\EDA.csv,Standard,ROUEN
6,0507BDS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-07-BD-S\V2\MONTRE\EDA - Copie.csv,Standard,ROUEN
7,0508SRR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-08-SR-R\V2\MONTRE\EDA.csv,Réduction de la menace,ROUEN
8,0509LGS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-09-LC-S\V2\MONTRE\EDA.csv,Standard,ROUEN
9,0510DFS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-10-DF-S\V2\MONTRE\1582188586_A012E9\EDA.csv,Standard,ROUEN


## LAVERAN

In [18]:
# ===== Racine LAVERAN =====
racine_laveran = (
    r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie"
    r"\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran"
)

# ===== Candidats attendus =====
candidats_laveran = [c.upper() for c in resultats_globaux["LAVERAN"]["valides"]]

# ===== Dictionnaire Numero_inclusion → Condition =====
df_conditions = resultats_globaux["LAVERAN"]["df"][["Numero_inclusion", "Condition"]].copy()
df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()

dict_condition = dict(
    zip(df_conditions["Numero_inclusion"], df_conditions["Condition"])
)

rows = []

# =========================================================
# BOUCLE PRINCIPALE
# =========================================================
for root, dirs, files in os.walk(racine_laveran):

    if "V2" not in root.upper():
        continue

    for f in files:

        # ===== CSV EDA uniquement =====
        if not f.lower().endswith(".csv"):
            continue
        if "EDA" not in f.upper():
            continue

        chemin = os.path.join(root, f)

        # ===== Numero_inclusion = 1er dossier après la racine =====
        rel_path = os.path.relpath(chemin, racine_laveran)
        pid = rel_path.split(os.sep)[0].upper()

        if pid not in candidats_laveran:
            continue

        rows.append({
            "Numero_inclusion": pid,
            "Chemin": chemin,
            "Condition": dict_condition.get(pid),
            "Feuille": "LAVERAN"
        })

        print(f"OK [{pid}] :", chemin)

# =========================================================
    # DATAFRAME + SUPPRESSION DES DOUBLONS
# =========================================================
df_laveran = pd.DataFrame(rows)

df_laveran_final = (
    df_laveran
    .sort_values("Chemin")
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .reset_index(drop=True)
)

# =========================================================
# AFFICHAGE
# =========================================================
print("\n====================================")
print(f"Candidats attendus LAVERAN : {len(candidats_laveran)}")
print(f"Fichiers EDA LAVERAN retenus : {len(df_laveran_final)}")

manquants = sorted(
    set(candidats_laveran) - set(df_laveran_final["Numero_inclusion"])
)

if manquants:
    print("⚠️ Candidats LAVERAN manquants :", manquants)
else:
    print("✅ Tous les candidats LAVERAN sont présents")

display(df_laveran_final)


OK [0626MCS] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0626MCS\V2\montre\1579594311_A01093\EDA.csv
OK [0628DMR] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0628DMR\V2\montre\1580375472_A01093\EDA.csv
OK [0629TCR] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0629TCR\V2\montre\1580805647_A01093\EDA.csv
OK [0630RJR] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0630RJR\0630RJR_V2\montre\1582190668_A01093\EDA.csv
OK [0631LHR] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0631LHR\V2\montre\1582624540_A01093\EDA.csv

Candidats attendus LAVERAN : 5
Fichiers EDA LAVERAN retenus : 5
✅ Tous les candidats LAVERAN sont présents


,Numero_inclusion,Chemin,Condition,Feuille
0,0626MCS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0626MCS\V2\montre\1579594311_A01093\EDA.csv,Standard,LAVERAN
1,0628DMR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0628DMR\V2\montre\1580375472_A01093\EDA.csv,Réduction de la menace,LAVERAN
2,0629TCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0629TCR\V2\montre\1580805647_A01093\EDA.csv,Réduction de la menace,LAVERAN
3,0630RJR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0630RJR\0630RJR_V2\montre\1582190668_A01093\EDA.csv,Réduction de la menace,LAVERAN
4,0631LHR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0631LHR\V2\montre\1582624540_A01093\EDA.csv,Réduction de la menace,LAVERAN


## LPC

In [20]:
# ===== Racine LPC =====
racine_lpc = (
    r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie"
    r"\LNSC\DESHAYES\DESHAYES\DATA_Aging"
)

# ===== Candidats attendus =====
candidats_lpc = [c.upper() for c in resultats_globaux["LPC"]["valides"]]

# ===== CORRECTIONS MANUELLES ARBORESCENCE → EXCEL =====
CORRESPONDANCE_IDS = {
    "0117MSR": "0117MMR" 
}
CORRESPONDANCE_INVERSE = {v: k for k, v in CORRESPONDANCE_IDS.items()}

# ===== Dictionnaire Numero_inclusion → Condition =====
df_conditions = resultats_globaux["LPC"]["df"][["Numero_inclusion", "Condition"]].copy()
df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()
dict_condition = dict(zip(df_conditions["Numero_inclusion"], df_conditions["Condition"]))

rows = []

# =========================================================
# BOUCLE PRINCIPALE
# =========================================================
for root, dirs, files in os.walk(racine_lpc):

    if "V2" not in root.upper():
        continue

    for f in files:

        if not f.lower().endswith(".csv"):
            continue
        if "EDA" not in f.upper():
            continue

        chemin = os.path.join(root, f)

        rel_path = os.path.relpath(chemin, racine_lpc)
        pid_arbo = rel_path.split(os.sep)[0].upper()

        pid_excel = CORRESPONDANCE_INVERSE.get(pid_arbo, pid_arbo)

        if pid_excel not in candidats_lpc:
            continue

        # =================================================
        # DEBUG TEMPS : UNIX (PATH) vs DATE FICHIER
        # =================================================
        try:
            dossier_montre = os.path.basename(os.path.dirname(chemin))
            ts_unix = int(dossier_montre.split("_")[0])
            date_unix = datetime.datetime.fromtimestamp(ts_unix)
        except Exception:
            date_unix = None

        date_fichier = datetime.datetime.fromtimestamp(
            os.path.getmtime(chemin)
        )

        print(
            f"🕒 {pid_excel} | "
            f"UNIX path = {date_unix} | "
            f"CSV modif = {date_fichier}"
        )

        # =================================================
        # ENREGISTREMENT
        # =================================================
        rows.append({
            "Numero_inclusion": pid_excel,
            "Chemin": chemin,
            "Condition": dict_condition.get(pid_excel),
            "Feuille": "LPC"
        })

        print(f"OK [{pid_excel}] :", chemin)

# =========================================================
# DATAFRAME + SUPPRESSION DES DOUBLONS
# =========================================================
df_lpc_final = (
    pd.DataFrame(rows)
    .sort_values("Chemin")
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .reset_index(drop=True)
)

# =========================================================
# AFFICHAGE
# =========================================================
print("\n====================================")
print(f"Candidats attendus LPC : {len(candidats_lpc)}")
print(f"Fichiers EDA LPC retenus : {len(df_lpc_final)}")

manquants = sorted(set(candidats_lpc) - set(df_lpc_final["Numero_inclusion"]))
if manquants:
    print("⚠️ Candidats LPC manquants :", manquants)
else:
    print("✅ Tous les candidats LPC sont présents")

display(df_lpc_final)


🕒 0101EMS | UNIX path = None | CSV modif = 2026-01-19 12:23:02.562823
OK [0101EMS] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0101EMS\V2\Montre\EDA.csv
🕒 0101EMS | UNIX path = 2019-01-22 10:10:40 | CSV modif = 2026-02-19 10:52:16.145401
OK [0101EMS] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0101EMS\V2\Montre\1548148240_A01115\EDA.csv
🕒 0103BPS | UNIX path = None | CSV modif = 2026-01-19 12:23:01.787221
OK [0103BPS] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0103BPS\V2\Montre\EDA.csv
🕒 0103BPS | UNIX path = 2018-12-11 10:08:34 | CSV modif = 2026-02-19 10:52:16.256335
OK [0103BPS] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0103BPS\V2\Montre\1544519314_A01115\EDA.csv
🕒 0104IBS | UNIX path = None | CSV modif = 2026-01-19 12:23:01.201204
OK [0104IBS] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\D

,Numero_inclusion,Chemin,Condition,Feuille
0,0101EMS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0101EMS\V2\Montre\1548148240_A01115\EDA.csv,Standard,LPC
1,0103BPS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0103BPS\V2\Montre\1544519314_A01115\EDA.csv,Standard,LPC
2,0104IBS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0104IBS\V2\Montre\1543395529_A01115\EDA.csv,Standard,LPC
3,0106DJR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0106DJR\V2\Montre\1545210317_A01115\EDA.csv,Réduction de la menace,LPC
4,0109MJR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0109MJR\V2\Montre\1544690346_A01115\EDA.csv,Réduction de la menace,LPC
5,0110RMS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0110RMS\V2\Montre\1544776620_A01115\EDA.csv,Standard,LPC
6,0113DGR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0113DGR\V2\Montre\1543308050_A01115\EDA.csv,Réduction de la menace,LPC
7,0114MLR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0114MLR\V2\Montre\1546851728_A01115\EDA.csv,Réduction de la menace,LPC
8,0115MHR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0115MHR\V2\Montre\1547196083_A01115\EDA.csv,Réduction de la menace,LPC
9,0116CNR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0116CNR\V2\Montre\1547109971_A01115\EDA.csv,Réduction de la menace,LPC


In [21]:
# ===== Racines =====
racines = [
    r"C:\Users\judupont\Desktop\data_aging_11_2025_copie\LNSC\LNSC\DESHAYES\DATA_Ancillaire",
    r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo"
]

# ===== Candidats LPC partie 2 =====
candidats_lpc2 = [
    c.upper() for c in resultats_globaux["LPC"]["valides"][24:]
]

# ===== Dictionnaire PID → Condition =====
df_conditions = resultats_globaux["LPC"]["df"][["Numero_inclusion", "Condition"]].copy()
df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()

dict_condition = dict(
    zip(df_conditions["Numero_inclusion"], df_conditions["Condition"])
)

rows = []

# =========================================================
# BOUCLE PRINCIPALE
# =========================================================
for racine in racines:

    print(f"\n--- Scan de : {racine}")

    for root, dirs, files in os.walk(racine):

        root_upper = root.upper()

        if not re.search(r'(^|[^A-Z0-9])V2([^A-Z0-9]|$)', root_upper):
            continue

        if "MONTRE" not in root_upper:
            continue

        for f in files:

            # ===== uniquement EDA.csv =====
            if f.upper() != "EDA.CSV":
                continue

            chemin = os.path.join(root, f)
            chemin_upper = chemin.upper()

            # ===== identification PID =====
            pid_trouve = None
            for c in candidats_lpc2:
                if c in chemin_upper:
                    pid_trouve = c
                    break

            if not pid_trouve:
                print("❌ PID NON TROUVÉ :", chemin)
                continue

            # =================================================
            # DEBUG TEMPS : UNIX (dossier) vs DATE fichier
            # =================================================
            dossier_parent = os.path.basename(os.path.dirname(chemin))
            ts_unix = None
            date_unix_str = "None"

            if "_" in dossier_parent:
                try:
                    ts_unix = int(dossier_parent.split("_")[0])
                    date_unix_str = datetime.datetime.fromtimestamp(
                        ts_unix
                    ).strftime("%Y-%m-%d %H:%M:%S")
                except Exception:
                    pass

            date_fichier_str = datetime.datetime.fromtimestamp(
                os.path.getmtime(chemin)
            ).strftime("%Y-%m-%d %H:%M:%S")

            print(
                f"🕒 PID={pid_trouve} | "
                f"Dossier='{dossier_parent}' | "
                f"UNIX path={date_unix_str} | "
                f"CSV modif={date_fichier_str}"
            )

            rows.append({
                "Numero_inclusion": pid_trouve,
                "Chemin": chemin,
                "Condition": dict_condition.get(pid_trouve),
                "Feuille": "LPC"
            })

            print(f"OK [{pid_trouve}] :", chemin)

# =========================================================
# DATAFRAME + SUPPRESSION DES DOUBLONS
# =========================================================
print("\nDEBUG rows length :", len(rows))

if rows:
    df_lpc2_final = (
        pd.DataFrame(rows)
        .sort_values("Chemin")
        .drop_duplicates(subset="Numero_inclusion", keep="first")
        .reset_index(drop=True)
    )
else:
    df_lpc2_final = pd.DataFrame(
        columns=["Numero_inclusion", "Chemin", "Condition", "Feuille"]
    )

# =========================================================
# AFFICHAGE
# =========================================================
print("\n====================================")
print(f"Candidats attendus LPC (partie 2) : {len(candidats_lpc2)}")
print(f"Fichiers EDA retenus              : {len(df_lpc2_final)}")

manquants = sorted(
    set(candidats_lpc2) - set(df_lpc2_final["Numero_inclusion"])
)

if manquants:
    print("⚠️ Candidats LPC manquants :", manquants)
else:
    print("✅ Tous les candidats LPC partie 2 sont présents")

display(df_lpc2_final)



--- Scan de : C:\Users\judupont\Desktop\data_aging_11_2025_copie\LNSC\LNSC\DESHAYES\DATA_Ancillaire

--- Scan de : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0101EMS\V2\Montre\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0101EMS\V2\Montre\1548148240_A01115\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0102BJS\V2\Montre\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0102BJS\V2\Montre\1545125104_A01115\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participan

,Numero_inclusion,Chemin,Condition,Feuille
0,0132GAR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0132GAR\V2\Montre\1582102629_A01115\EDA.csv,Réduction de la menace,LPC
1,0133BPS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0133BPS-0157BPS\V2\Montre\1582707829_A01115\EDA.csv,Standard,LPC
2,0134IFR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0134IFR\V2\montre\1583224544_A01115\EDA.csv,Réduction de la menace,LPC
3,0135KLS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0135KLS\0135KLS_0163KLS_V2\montre\1614757320_A01115\EDA.csv,Standard,LPC
4,0136CJR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0136CJR\0136CJR-V2\MONTRE\1617350469_A01115\EDA.csv,Réduction de la menace,LPC
5,0137BMS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0137BMS-V2\MONTRE\1618214285_A01115 (1)\EDA.csv,Standard,LPC
6,0138NWR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0138NWR\0138NWR-V2\montre\1622621441_A01115\EDA.csv,Réduction de la menace,LPC
7,0144ZGR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0144ZGR\0144ZGR_V2\montre\1656489854_A01115\EDA.csv,Réduction de la menace,LPC
8,0140LMR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0168LMR 0140LMR\V2 0140LMR\MONTRE\1642063125_A01115\EDA.csv,Réduction de la menace,LPC
9,0143EBR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0171EBR _ 0143EBR\0143EBR V2\montre\1651050086_A01115\EDA.csv,Réduction de la menace,LPC


## SAINTE-MARGUERITE

In [22]:
# ===== Racine Sainte-Marguerite =====
racine_sm = r"C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite"

# ===== Candidats attendus =====
candidats_sm = [c.upper() for c in resultats_globaux["SAINTE-MARGUERITE"]["valides"]]

# ===== Dictionnaire PID → Condition =====
df_conditions = resultats_globaux["SAINTE-MARGUERITE"]["df"][["Numero_inclusion", "Condition"]].copy()
df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()
dict_condition = dict(zip(df_conditions["Numero_inclusion"], df_conditions["Condition"]))

# ===== Liste pour stocker les fichiers EDA =====
rows = []

# =========================================================
# BOUCLE SUR LES ZIP avec filtre V2
# =========================================================
for root, dirs, files in os.walk(racine_sm):

    if "V2" not in root.upper():
        continue

    for f in files:

        if not f.lower().endswith(".zip"):
            continue

        chemin_zip = os.path.join(root, f)
        chemin_upper = chemin_zip.upper()

        # ===== Cherche Numer_inclusion comme sous-chaîne =====
        pid_trouve = None
        for c in candidats_sm:
            if c in chemin_upper:
                pid_trouve = c
                break

        if not pid_trouve:
            print(f"❌ PID NON TROUVÉ : {chemin_zip}")
            continue

        # ===== Dézipper dans le dossier du ZIP lui-même =====
        try:
            with zipfile.ZipFile(chemin_zip, 'r') as zf:
                # Cherche uniquement les fichiers EDA CSV
                eda_files = [name for name in zf.namelist() if "EDA" in name.upper() and name.lower().endswith(".csv")]

                if not eda_files:
                    print(f"❌ Pas de fichier EDA dans : {chemin_zip}")
                    continue

                for eda_name in eda_files:
                    # Extraction directement dans le dossier contenant le ZIP
                    zf.extract(eda_name, root)
                    eda_path = os.path.join(root, eda_name)

                    rows.append({
                        "Numero_inclusion": pid_trouve,
                        "Chemin": eda_path,
                        "Condition": dict_condition.get(pid_trouve),
                        "Feuille": "SAINTE-MARGUERITE"
                    })

                    print(f"✅ EDA extrait [{pid_trouve}] :", eda_path)

        except zipfile.BadZipFile:
            print(f"❌ ZIP corrompu : {chemin_zip}")

# =========================================================
# DATAFRAME FINAL
# =========================================================
df_sm = pd.DataFrame(rows)

# Suppression des doublons
df_sm = df_sm.drop_duplicates(subset="Numero_inclusion", keep="first").reset_index(drop=True)

print(f"\nNombre de fichiers EDA extraits Sainte-Marguerite : {len(df_sm)}")
display(df_sm)

✅ EDA extrait [0801HDR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0801HDR\0801HDR - V2\MONTRE_0801HDR_V2\EDA.csv
✅ EDA extrait [0802LAS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0802LAS\0802LAS - V2\MONTRE_0802LAS_V2\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0803DPS\0803DPS - V2\MONTRE_0803DPS_V2\1650010647_A01115(1).zip
✅ EDA extrait [0804GOR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0804GOR\0804GOR - V2\MONTRE_0804GOR_V2\EDA.csv
✅ EDA extrait [0805BMS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0805BMS\0805BMS - V2\MONTRE_0805BMS_V2\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TO

,Numero_inclusion,Chemin,Condition,Feuille
0,0801HDR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0801HDR\0801HDR - V2\MONTRE_0801HDR_V2\EDA.csv,Réduction de la menace,SAINTE-MARGUERITE
1,0802LAS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0802LAS\0802LAS - V2\MONTRE_0802LAS_V2\EDA.csv,Standard,SAINTE-MARGUERITE
2,0804GOR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0804GOR\0804GOR - V2\MONTRE_0804GOR_V2\EDA.csv,Réduction de la menace,SAINTE-MARGUERITE
3,0805BMS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0805BMS\0805BMS - V2\MONTRE_0805BMS_V2\EDA.csv,Standard,SAINTE-MARGUERITE
4,0807OMR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0807OMR\V2\0807OMR - V2\MONTRE_0807OMR_V2\EDA.csv,Réduction de la menace,SAINTE-MARGUERITE
5,0808PJR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0808PJR\0808PJR-V2\montre\EDA.csv,Réduction de la menace,SAINTE-MARGUERITE


## CGD

In [24]:
# ===== Racine CGD =====
racine_cgd = (
    r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie"
    r"\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD"
)

# ===== Candidats attendus =====
candidats_cgd = [
    c.upper() for c in resultats_globaux["CGD"]["valides"]
]

# ===== Dictionnaire Numero_inclusion → Condition =====
df_conditions = resultats_globaux["CGD"]["df"][
    ["Numero_inclusion", "Condition"]
].copy()

df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()

dict_condition = dict(
    zip(df_conditions["Numero_inclusion"], df_conditions["Condition"])
)

rows_cgd = []

# =========================================================
# BOUCLE PRINCIPALE
# =========================================================
for root, dirs, files in os.walk(racine_cgd):

    if "V2" not in root.upper():
        continue

    for f in files:

        if not f.lower().endswith(".csv"):
            continue
        if "EDA" not in f.upper():
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        # ===== identification =====
        pid = None
        for c in candidats_cgd:
            if c in chemin_upper:
                pid = c
                break

        if not pid:
            continue

        rows_cgd.append({
            "Numero_inclusion": pid,
            "Chemin": chemin,
            "Condition": dict_condition.get(pid),
            "Feuille": "CGD"
        })

        print(f"OK [{pid}] :", chemin)

# =========================================================
# DATAFRAME + SUPPRESSION DES DOUBLONS
# =========================================================
df_cgd = pd.DataFrame(rows_cgd)

df_cgd = (
    df_cgd
    .sort_values("Chemin")
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .reset_index(drop=True)
)

# =========================================================
# AFFICHAGE
# =========================================================
print("\n====================================")
print(f"Candidats attendus CGD : {len(candidats_cgd)}")
print(f"Fichiers EDA retenus   : {len(df_cgd)}")

manquants = sorted(
    set(candidats_cgd) - set(df_cgd["Numero_inclusion"])
)

if manquants:
    print("⚠️ Candidats manquants :", manquants)
else:
    print("✅ Tous les candidats CGD sont présents")

display(df_cgd)


OK [0902SIS] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0902SIS\0902SIS_V2\MONTRE 0102SIS_V2\1652946369_A01115(1)\EDA.csv
OK [0904KJS] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0904KJS\0904KJS - V2\MONTRE_0904KJS_V2\1656317741_A01115(3)\EDA.csv
OK [0905SLR] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0905SLR\0905SLR-V2\montre\1656922153_A01115\EDA.csv
OK [0906DNR] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0906DNR\0906DNR_V2\montre\1657095308_A01115\EDA.csv

Candidats attendus CGD : 5
Fichiers EDA retenus   : 4
⚠️ Candidats manquants : ['0901SMR']


,Numero_inclusion,Chemin,Condition,Feuille
0,0902SIS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0902SIS\0902SIS_V2\MONTRE 0102SIS_V2\1652946369_A01115(1)\EDA.csv,Standard,CGD
1,0904KJS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0904KJS\0904KJS - V2\MONTRE_0904KJS_V2\1656317741_A01115(3)\EDA.csv,Standard,CGD
2,0905SLR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0905SLR\0905SLR-V2\montre\1656922153_A01115\EDA.csv,Réduction de la menace,CGD
3,0906DNR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0906DNR\0906DNR_V2\montre\1657095308_A01115\EDA.csv,Réduction de la menace,CGD


## CERCA

In [26]:
# ===== Racine CERCA =====
racine_cerca = r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS"

# ===== Candidats CERCA =====
candidats_cerca = [c.upper() for c in resultats_globaux['CERCA']['valides']]

# ===== Dictionnaire PID → Condition =====
df_conditions = resultats_globaux['CERCA']['df'][["Numero_inclusion", "Condition"]].copy()
df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()
dict_condition = dict(zip(df_conditions["Numero_inclusion"], df_conditions["Condition"]))

# ===== Liste pour stocker les fichiers EDA =====
rows = []

# =========================================================
# BOUCLE PRINCIPALE
# =========================================================
for root, dirs, files in os.walk(racine_cerca):

    if "V2" not in root.upper():
        continue

    for f in files:

        if not f.lower().endswith(".csv"):
            continue
        if "EDA" not in f.upper():
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        # ===== Identification =====
        pid_trouve = None
        for c in candidats_cerca:
            if c in chemin_upper:
                pid_trouve = c
                break

        if not pid_trouve:
            print(f"❌ PID NON TROUVÉ : {chemin}")
            continue

        rows.append({
            "Numero_inclusion": pid_trouve,
            "Chemin": chemin,
            "Condition": dict_condition.get(pid_trouve),
            "Feuille": "CERCA"
        })

        print(f"✅ EDA trouvé [{pid_trouve}] :", chemin)

# =========================================================
# DATAFRAME 
# =========================================================
df_cerca_final = pd.DataFrame(rows)
df_cerca_final = df_cerca_final.drop_duplicates(subset="Numero_inclusion", keep="first").reset_index(drop=True)

# =========================================================
# AFFICHAGE
# =========================================================
print("\n====================================")
print(f"Candidats attendus CERCA : {len(candidats_cerca)}")
print(f"Fichiers EDA retenus CERCA : {len(df_cerca_final)}")

manquants = sorted(set(candidats_cerca) - set(df_cerca_final["Numero_inclusion"]))
if manquants:
    print("⚠️ Candidats manquants :", manquants)
else:
    print("✅ Tous les candidats CERCA sont présents")

display(df_cerca_final)

❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0401BDS\V2\Montre\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0402TFR\V2\Montre\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0405RCR\V2\Montre\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0406BBS\V2\Montre\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0408SJS\V2\Montre\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0411NPR\V2\Montre\EDA.csv
✅ EDA trouvé [0413MMR] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0413MMR\V2\M

,Numero_inclusion,Chemin,Condition,Feuille
0,0413MMR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0413MMR\V2\Montre\EDA.csv,Réduction de la menace,CERCA
1,0415HJS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0415HJS\V2\Montre\EDA.csv,Standard,CERCA
2,0417FAS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0417FAS\V2\Montre\EDA.csv,Standard,CERCA
3,0418MPR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0418MPR\V2\Montre\EDA.csv,Réduction de la menace,CERCA
4,0420MCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0420MCR\V2\Montre\EDA.csv,Réduction de la menace,CERCA
5,0421ACS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0421ACS\V2\Montre\EDA.csv,Standard,CERCA
6,0414PVR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Camille GUILLOU\0414PVR\V2\Montre\EDA.csv,Réduction de la menace,CERCA


## CONCATENATION

In [27]:
# ===== Liste des DataFrames par feuille =====
dfs = [
    df_aphm_final,  
    df_caen_final,
    df_poitiers_final,     
    df_rouen_final,
    df_laveran_final,
    df_lpc_final,         
    df_lpc2_final,
    df_sm,          
    df_cgd,         
    df_cerca_final      
]

# ===== Normalisation des colonnes =====
for i, df in enumerate(dfs):
    if "PID" in df.columns:
        df.rename(columns={"PID": "Numero_inclusion"}, inplace=True)
    # Assurer que la colonne Condition existe
    if "Condition" not in df.columns:
        df["Condition"] = None
    # On garde uniquement les colonnes essentielles
    df = df[["Numero_inclusion", "Chemin", "Feuille", "Condition"]]
    dfs[i] = df

# ===== Concatenation =====
df_global = pd.concat(dfs, ignore_index=True)

# ===== Suppression des doublons par Numero_inclusion =====
# On garde le premier fichier détecté par candidat (priorité ceinture déjà appliquée)
df_global = df_global.drop_duplicates(subset=["Numero_inclusion"], keep="first")

# ===== Tri par Numero_inclusion =====
df_global = df_global.sort_values("Numero_inclusion").reset_index(drop=True)

# ===== Résumé =====
print(f"Total candidats uniques : {df_global['Numero_inclusion'].nunique()}")
print(f"Total fichiers conservés : {len(df_global)}")
display(df_global)

Total candidats uniques : 156
Total fichiers conservés : 156


,Numero_inclusion,Chemin,Feuille,Condition
0,0101CAR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0101CAR\0101CAR 06-07-2018 V2\Montre 0101CAR\1530858300_A01093\EDA.csv,APHM,Réduction de la menace
1,0101EMS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0101EMS\V2\Montre\1548148240_A01115\EDA.csv,LPC,Standard
2,0102PCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0102PCR\0102PCR 09-07-2018 V2\Montre 0102PCR\1531119483_A01093\EDA.csv,APHM,Réduction de la menace
3,0103BPS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0103BPS\V2\Montre\1544519314_A01115\EDA.csv,LPC,Standard
4,0103SHS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0103SHS\0103SHS 10-09-2018 V2\Montre 0103SHS\1536563089_A01093\EDA.csv,APHM,Standard
5,0104FJS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0104FJS\0104FJS 29-11-2018 V2\Montre 0104FJS\1543476632_A01093\EDA.csv,APHM,Standard
6,0104IBS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0104IBS\V2\Montre\1543395529_A01115\EDA.csv,LPC,Standard
7,0105PNR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0105PNR\0105PNR 10-12-2018 V2\Montre 0105PNR\1544426613_A01093\EDA.csv,APHM,Réduction de la menace
8,0106DJR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0106DJR\V2\Montre\1545210317_A01115\EDA.csv,LPC,Réduction de la menace
9,0106JLS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0106JLS\0106JLS 03-01-2019 V2\Montre 0106JLS\1546500427_A01093\EDA.csv,APHM,Standard


In [28]:
# CONSERVATION DES FEUILLES ET DES CONDITIONS 

chemin_txt = r"C:\Users\judupont\Desktop\df_global_v2_EDA.txt"

df_global.to_csv(
    chemin_txt,
    sep="\t",
    index=False,
    encoding="utf-8-sig"
)

print(f"📄 df_global sauvegardé : {chemin_txt}")

📄 df_global sauvegardé : C:\Users\judupont\Desktop\df_global_v2_EDA.txt


# Verification que le temps d'experience et le temps du fichier EDA correspondent, offset et continuité

In [29]:
def convertir_heure_en_secondes(x):
    if pd.isna(x):
        return None

    if isinstance(x, (int, float)):
        seconds = float(x) * 24 * 3600
        return seconds % (24 * 3600)

    try:
        t = pd.to_datetime(str(x).strip()).time()
        return t.hour * 3600 + t.minute * 60 + t.second
    except Exception:
        return None

set_global = set(df_global["Numero_inclusion"])


dfs_excel = []

for feuille, data in resultats_bloc1.items():

    df_excel = data["df"].copy()

    # ===== NORMALISATION EXCEL =====
    df_excel["Numero_inclusion"] = (
        df_excel["Numero_inclusion"]
        .astype(str)
        .str.strip()
        .str.upper()
    )
    
    df_excel = df_excel[
        df_excel["Numero_inclusion"].isin(set_global)
    ]

    if df_excel.empty:
        continue

    # ===== Conversion heures =====
    df_excel["heure_debut_sec"] = df_excel["heure_montre_v2"].apply(convertir_heure_en_secondes)
    df_excel["heure_fin_sec"] = df_excel["heure_fin_tests_v2"].apply(convertir_heure_en_secondes)

    # ========= Durée de la visite ========== 
    df_excel["duree_experience_sec"] = (
        df_excel["heure_fin_sec"] - df_excel["heure_debut_sec"]
    )

    df_excel.loc[
        df_excel["duree_experience_sec"] < 0,
        "duree_experience_sec"
    ] += 24 * 3600

    df_excel["Feuille"] = feuille
    dfs_excel.append(df_excel)


if dfs_excel:
    df_excel_global = pd.concat(dfs_excel, ignore_index=True)
else:
    df_excel_global = pd.DataFrame()

print(f"Lignes Excel retenues : {len(df_excel_global)}")
display(df_excel_global[[
    "Numero_inclusion",
    "Feuille",
    "duree_experience_sec"
]])

Lignes Excel retenues : 156


,Numero_inclusion,Feuille,duree_experience_sec
0,0101CAR,APHM,9420
1,0102PCR,APHM,7560
2,0103SHS,APHM,5520
3,0105PNR,APHM,7260
4,0104FJS,APHM,9600
5,0106JLS,APHM,7440
6,0107DSS,APHM,7320
7,0108BFS,APHM,7920
8,0109GSS,APHM,7500
9,0110LPR,APHM,8160


In [30]:
# ==== Conversion Date dans Excel ====
df_excel_global["Date_v2_dt"] = pd.to_datetime(
    df_excel_global["Date_v2"], errors="coerce"
).dt.date

rows_eda = []

for _, row in df_global.iterrows():

    pid = str(row["Numero_inclusion"]).strip().upper()
    chemin = row["Chemin"]

    # ---- récupérer ligne Excel ----
    ligne_excel = df_excel_global[df_excel_global["Numero_inclusion"] == pid]
    if ligne_excel.empty:
        continue

    date_v2 = ligne_excel.iloc[0]["Date_v2_dt"]
    duree_excel_sec = ligne_excel.iloc[0]["duree_experience_sec"]

    # ---- lecture CSV EDA (ZIP ou non) ----
    try:
        if ".ZIP|" in chemin.upper():
            zip_path, inner_csv = chemin.split("|", 1)
            with zipfile.ZipFile(zip_path, "r") as z:
                with z.open(inner_csv) as f:
                    df_eda_raw = pd.read_csv(f, header=None)
        else:
            df_eda_raw = pd.read_csv(chemin, header=None)

    except Exception as e:
        print(f"⚠️ Lecture impossible EDA pour {pid} : {e}")
        continue

    if len(df_eda_raw) < 3:
        continue

    # ---- récupération timestamp initial + fréquence ----
    try:
        t0_unix = float(df_eda_raw.iloc[0, 0])
        freq = float(df_eda_raw.iloc[1, 0])
    except:
        continue

    if freq <= 0:
        continue

    # ---- données EDA ----
    df_eda = df_eda_raw.iloc[2:].copy()
    df_eda.columns = ["EDA"]
    df_eda["EDA"] = pd.to_numeric(df_eda["EDA"], errors="coerce")
    df_eda = df_eda.dropna()

    if df_eda.empty:
        continue

    # ---- construction timestamps ----
    pas = 1 / freq
    n_points = len(df_eda)

    duree_eda_sec = (n_points - 2) * pas

    # timestamp début/fin
    t_debut = datetime.datetime.fromtimestamp(t0_unix)
    t_fin = t_debut + datetime.timedelta(seconds=duree_eda_sec)


    rows_eda.append({
        "Numero_inclusion": pid,
        "duree_eda_sec": round(duree_eda_sec, 0),
        "duree_excel_sec": duree_excel_sec,
        "nb_points": n_points,
        "frequence": freq
    })

df_eda_global = pd.DataFrame(rows_eda)

print(f"EDA calculés : {len(df_eda_global)}")
display(df_eda_global)


EDA calculés : 156


,Numero_inclusion,duree_eda_sec,duree_excel_sec,nb_points,frequence
0,0101CAR,9529.0,9420,38118,4.0
1,0101EMS,6298.0,6240,25194,4.0
2,0102PCR,7702.0,7560,30810,4.0
3,0103BPS,5198.0,5160,20796,4.0
4,0103SHS,5628.0,5520,22512,4.0
5,0104FJS,9864.0,9600,39456,4.0
6,0104IBS,5665.0,5640,22662,4.0
7,0105PNR,7476.0,7260,29904,4.0
8,0106DJR,6658.0,6600,26634,4.0
9,0106JLS,7638.0,7440,30552,4.0


In [31]:
global_ids = set(df_global["Numero_inclusion"])
excel_ids = set(df_excel_global["Numero_inclusion"])

manquants_dans_global = excel_ids - global_ids
manquants_dans_excel = global_ids - excel_ids

print("Présents dans Excel mais pas dans RR :", manquants_dans_global)
print("Présents dans RR mais pas dans Excel :", manquants_dans_excel)

Présents dans Excel mais pas dans RR : set()
Présents dans RR mais pas dans Excel : set()


# Verif temps unix

In [ ]:
# Exemple de timestamp Unix
timestamp_unix = 1530858300.000000

# Conversion en datetime
dt = datetime.datetime.fromtimestamp(timestamp_unix)

print(dt)

# D'ou vient l'écart ? L'offest doit etre appliqué en debut, en fin, au milieu?

In [32]:
def sec_to_datetime(sec, date_ref):
    return datetime.combine(
        date_ref.date(),
        datetime.min.time()
    ) + timedelta(seconds=int(sec))


resultats_offset_eda = []

for _, row in df_global.iterrows():

    pid = str(row["Numero_inclusion"]).strip().upper()
    chemin_eda = row["Chemin"]

    # ===== Lecture EDA =====
    try:
        if ".ZIP|" in chemin_eda.upper():
            zip_path, inner_csv = chemin_eda.split("|", 1)
            with zipfile.ZipFile(zip_path, "r") as z:
                with z.open(inner_csv) as f:
                    df_raw = pd.read_csv(f, header=None)
        else:
            df_raw = pd.read_csv(chemin_eda, header=None)

    except Exception as e:
        print(f"⚠️ Lecture EDA impossible pour {pid} : {e}")
        continue

    if len(df_raw) < 3:
        continue

    # ===== Reconstruction Timestamp =====
    try:
        t0_unix = float(df_raw.iloc[0, 0])
        freq = float(df_raw.iloc[1, 0])
    except:
        continue

    if freq <= 0:
        continue

    df_eda = df_raw.iloc[2:].copy()
    df_eda.columns = ["EDA"]
    df_eda["EDA"] = pd.to_numeric(df_eda["EDA"], errors="coerce")
    df_eda = df_eda.dropna()

    if df_eda.empty:
        continue

    pas = 1 / 4
    n = len(df_eda)

    t_debut = datetime.fromtimestamp(t0_unix)
    df_eda["Timestamp"] = [
        t_debut + timedelta(seconds=i * pas)
        for i in range(n)
    ]

    t_min = df_eda["Timestamp"].min()
    t_max = df_eda["Timestamp"].max()

    ligne_excel = df_excel_global.loc[
        df_excel_global["Numero_inclusion"] == pid
    ]

    if ligne_excel.empty:
        continue

    h_debut_sec = ligne_excel["heure_debut_sec"].values[0]
    h_fin_sec   = ligne_excel["heure_fin_sec"].values[0]
    duree_excel_sec = ligne_excel["duree_experience_sec"].values[0]

    if pd.isna(h_debut_sec) or pd.isna(h_fin_sec) or pd.isna(duree_excel_sec):
        continue

    h_debut = sec_to_datetime(h_debut_sec, t_min)
    h_fin   = sec_to_datetime(h_fin_sec, t_min)

    # =====================================================
    # Calcul du retard si EDA commence après Excel
    # =====================================================
    if t_min > h_debut:
        delta_retard_sec = (t_min - h_debut).total_seconds()
    else:
        delta_retard_sec = 0

    # ===== Comptages =====
    nb_avant = (df_eda["Timestamp"] < h_debut).sum()
    nb_dans = (
        (df_eda["Timestamp"] >= h_debut) &
        (df_eda["Timestamp"] <= h_fin)
    ).sum()
    nb_apres = (df_eda["Timestamp"] > h_fin).sum()

    total = len(df_eda)

    if nb_apres > nb_avant:
        origine = "FIN"
    elif nb_avant > nb_apres:
        origine = "DEBUT"
    else:
        origine = "MIXTE"

    resultats_offset_eda.append({
        "Numero_inclusion": pid,
        "nb_total_points": total,
        "nb_avant_excel": nb_avant,
        "nb_dans_excel": nb_dans,
        "nb_apres_excel": nb_apres,
        "origine_offset": origine,
        "delta_retard_sec": delta_retard_sec
    })

df_offset_eda = pd.DataFrame(resultats_offset_eda)

print(f"Offsets EDA calculés : {len(df_offset_eda)}")
display(df_offset_eda)

Offsets EDA calculés : 156


,Numero_inclusion,nb_total_points,nb_avant_excel,nb_dans_excel,nb_apres_excel,origine_offset,delta_retard_sec
0,0101CAR,38118,0,37681,437,FIN,0.0
1,0101EMS,25194,0,24801,393,FIN,40.0
2,0102PCR,30810,0,30229,581,FIN,3.0
3,0103BPS,20796,0,20505,291,FIN,34.0
4,0103SHS,22512,0,21885,627,FIN,49.0
5,0104FJS,39456,0,38033,1423,FIN,92.0
6,0104IBS,22662,0,22365,297,FIN,49.0
7,0105PNR,29904,0,28909,995,FIN,33.0
8,0106DJR,26634,0,26093,541,FIN,77.0
9,0106JLS,30552,0,29733,819,FIN,7.0


In [34]:
# ===== Dossier de sortie =====
desktop = Path.home() / "Desktop"
output_dir = desktop / "EDA_v2_tronqués"
output_dir.mkdir(exist_ok=True)

print(f"📁 Dossier de sortie : {output_dir}")

# ===== Boucle sur les fichiers tronqués =====
for _, row in df_offset_eda.iterrows():

    pid = str(row["Numero_inclusion"]).strip().upper()

    # récupérer chemin EDA depuis df_global
    ligne_global = df_global[df_global["Numero_inclusion"] == pid]
    if ligne_global.empty:
        continue

    chemin_eda = ligne_global.iloc[0]["Chemin"]

    nb_avant = int(row["nb_avant_excel"])
    nb_apres = int(row["nb_apres_excel"])

    # ===== Lecture EDA (ZIP ou non) =====
    try:
        if ".ZIP|" in chemin_eda.upper():
            zip_path, inner_csv = chemin_eda.split("|", 1)

            with zipfile.ZipFile(zip_path, "r") as z:
                with z.open(inner_csv) as f:
                    df_eda = pd.read_csv(f)
        else:
            df_eda = pd.read_csv(chemin_eda)

    except Exception as e:
        print(f"❌ Lecture impossible {pid}: {e}")
        continue

    total = len(df_eda)

    # ===== Indices de coupe =====
    # On conserve les 2 premières lignes (temps UNIX et fréquence), troncage à partir de la 3ème
    start = 2 + nb_avant
    end = total - nb_apres

    if start >= end:
        print(f"⚠️ Troncature invalide pour {pid} (start={start}, end={end})")
        continue

    # On concatène les 2 premières lignes intactes + le reste tronqué
    df_eda_trunc = pd.concat([df_eda.iloc[:2], df_eda.iloc[start:end]]).reset_index(drop=True)

    if df_eda_trunc.empty:
        print(f"⚠️ {pid} → fichier vide après troncature")
        continue

    # ===== Nom fichier de sortie =====
    if ".ZIP|" in chemin_eda.upper():
        nom_base = Path(inner_csv).name
    else:
        nom_base = Path(chemin_eda).name

    nom_fichier = f"offset_{pid}_{nom_base}"
    path_sortie = output_dir / nom_fichier

    # ===== Sauvegarde =====
    df_eda_trunc.to_csv(path_sortie, index=False)

    print(
        f"✅ {pid} | "
        f"avant={nb_avant}, après={nb_apres} | "
        f"{len(df_eda_trunc)} lignes → {path_sortie.name}"
    )

print("🎯 Troncature terminée")

📁 Dossier de sortie : C:\Users\judupont\Desktop\EDA_v2_tronqués
✅ 0101CAR | avant=0, après=437 | 37682 lignes → offset_0101CAR_EDA.csv
✅ 0101EMS | avant=0, après=393 | 24802 lignes → offset_0101EMS_EDA.csv
✅ 0102PCR | avant=0, après=581 | 30230 lignes → offset_0102PCR_EDA.csv
✅ 0103BPS | avant=0, après=291 | 20506 lignes → offset_0103BPS_EDA.csv
✅ 0103SHS | avant=0, après=627 | 21886 lignes → offset_0103SHS_EDA.csv
✅ 0104FJS | avant=0, après=1423 | 38034 lignes → offset_0104FJS_EDA.csv
✅ 0104IBS | avant=0, après=297 | 22366 lignes → offset_0104IBS_EDA.csv
✅ 0105PNR | avant=0, après=995 | 28910 lignes → offset_0105PNR_EDA.csv
✅ 0106DJR | avant=0, après=541 | 26094 lignes → offset_0106DJR_EDA.csv
✅ 0106JLS | avant=0, après=819 | 29734 lignes → offset_0106JLS_EDA.csv
✅ 0107DSS | avant=0, après=275 | 28874 lignes → offset_0107DSS_EDA.csv
✅ 0108BFS | avant=0, après=715 | 31494 lignes → offset_0108BFS_EDA.csv
✅ 0109GSS | avant=0, après=1413 | 29842 lignes → offset_0109GSS_EDA.csv
✅ 0109MJR |

In [35]:
# ===== Paramètres =====
min_lignes = 10000
candidats_a_exclure = {"0356DMS","0130FAR"} # le fichier du 0130FAR est celui de la v4

dossier = Path.home() / "Desktop" / "EDA_v2_tronqués"

print(f"📂 Dossier analysé : {dossier}")

supprimes = []
conserves = []

for fichier in dossier.glob("*.csv"):

    nom = fichier.name

    # ===== Extraction PID depuis le nom =====
    match = re.search(r"\d{4}[A-Z]{3}", nom.upper())
    if not match:
        print(f"⚠️ PID introuvable dans {nom}")
        continue

    pid = match.group()

    # ===== Exclusion candidat spécifique =====
    if pid in candidats_a_exclure:
        fichier.unlink()
        supprimes.append((nom, "candidat exclu"))
        print(f"🗑️ {nom} → candidat exclu ({pid})")
        continue

    # ===== Comptage rapide des lignes =====
    try:
        with open(fichier, "r", encoding="utf-8") as f:
            n_lignes = sum(1 for _ in f)
    except Exception as e:
        print(f"❌ Lecture impossible {nom}: {e}")
        continue

    # ===== Exclusion fichiers trop courts =====
    if n_lignes < min_lignes:
        fichier.unlink()
        supprimes.append((nom, f"{n_lignes} lignes"))
        print(f"❌ {nom} → trop petit ({n_lignes} lignes)")
    else:
        conserves.append((nom, n_lignes))
        print(f"✅ {nom} → conservé ({n_lignes} lignes)")

# ===== Résumé =====
print("\n====== RÉSUMÉ ======")
print(f"Fichiers supprimés : {len(supprimes)}")
print(f"Fichiers conservés : {len(conserves)}")

if supprimes:
    print("\nDétail suppressions :")
    for s in supprimes:
        print(" -", s)


📂 Dossier analysé : C:\Users\judupont\Desktop\EDA_v2_tronqués
✅ offset_0101CAR_EDA.csv → conservé (37683 lignes)
✅ offset_0101EMS_EDA.csv → conservé (24803 lignes)
✅ offset_0102PCR_EDA.csv → conservé (30231 lignes)
✅ offset_0103BPS_EDA.csv → conservé (20507 lignes)
✅ offset_0103SHS_EDA.csv → conservé (21887 lignes)
✅ offset_0104FJS_EDA.csv → conservé (38035 lignes)
✅ offset_0104IBS_EDA.csv → conservé (22367 lignes)
✅ offset_0105PNR_EDA.csv → conservé (28911 lignes)
✅ offset_0106DJR_EDA.csv → conservé (26095 lignes)
✅ offset_0106JLS_EDA.csv → conservé (29735 lignes)
✅ offset_0107DSS_EDA.csv → conservé (28875 lignes)
✅ offset_0108BFS_EDA.csv → conservé (31495 lignes)
✅ offset_0109GSS_EDA.csv → conservé (29843 lignes)
✅ offset_0109MJR_EDA.csv → conservé (25304 lignes)
✅ offset_0110LPR_EDA.csv → conservé (31471 lignes)
✅ offset_0110RMS_EDA.csv → conservé (22563 lignes)
✅ offset_0111MNR_EDA.csv → conservé (30227 lignes)
✅ offset_0112BSR_EDA.csv → conservé (35919 lignes)
✅ offset_0113BMR_EDA

# On peut passer au découpage!

In [47]:
def decouper_eda_par_bloc(df_eda, duree_bloc1, duree_video, duree_bloc2, duree_bloc3, delta):
    
    df_eda = df_eda.copy()
    
    if len(df_eda) < 3:
        print("⚠️ Fichier trop court pour découpages")
        return None

    # ===== Lire timestamp initial et fréquence =====
    ts_unix = df_eda.iloc[0, 0]  # première ligne, première colonne
    freq_hz = df_eda.iloc[1, 0]  # deuxième ligne, première colonne
    print(df_eda.iloc[0])
    print(df_eda.iloc[1])

    try:
        t0 = datetime.datetime.fromtimestamp(int(ts_unix))
        delta_sec = 1.0 / 4
    except Exception as e:
        print(f"⚠️ Impossible de lire timestamp/freq : {e}")
        return None

    # ===== Générer colonne Timestamp =====
    nb_lignes = len(df_eda) - 2  # exclure les 2 premières lignes
    timestamps = [t0 + datetime.timedelta(seconds=i * delta_sec) for i in range(nb_lignes)]
    df_eda_data = df_eda.iloc[2:].copy()
    df_eda_data["Timestamp"] = timestamps

    t_fin_rr = df_eda_data["Timestamp"].max()

    # ===== Bornes temporelles =====
    fin_bloc1 = t0 + pd.Timedelta(seconds=max(0, duree_bloc1 - delta))
    debut_bloc2 = fin_bloc1 + pd.Timedelta(seconds=duree_video)
    fin_bloc2 = debut_bloc2 + pd.Timedelta(seconds=duree_bloc2)
    fin_bloc3 = fin_bloc2 + pd.Timedelta(seconds=duree_bloc3)
    fin_bloc3 = min(fin_bloc3, t_fin_rr)  # sécurité

    # ===== Découpage =====
    blocs = {
        "bloc1": df_eda_data[(df_eda_data["Timestamp"] >= t0) & (df_eda_data["Timestamp"] < fin_bloc1)],
        "bloc2": df_eda_data[(df_eda_data["Timestamp"] >= debut_bloc2) & (df_eda_data["Timestamp"] < fin_bloc2)],
        "bloc3": df_eda_data[(df_eda_data["Timestamp"] >= fin_bloc2) & (df_eda_data["Timestamp"] <= fin_bloc3)],
        "bornes": {
            "t0": t0,
            "fin_bloc1": fin_bloc1,
            "debut_bloc2": debut_bloc2,
            "fin_bloc2": fin_bloc2,
            "fin_bloc3": fin_bloc3,
            "t_fin_rr": t_fin_rr
        }
    }

    return blocs


In [44]:
def convertir_et_normaliser_heure(x):
    """
    Convertit n'importe quel format d'heure Excel / texte / datetime en pd.Timestamp
    normalisé sur le 1900-01-01, ne gardant que l'heure, minute, seconde.
    """
    if pd.isna(x):
        return pd.NaT

    # Excel numérique → fraction de jour ou date complète
    if isinstance(x, (int, float)):
        seconds = (float(x) * 24 * 3600) % (24*3600)
        return pd.Timestamp("1900-01-01") + pd.to_timedelta(seconds, unit="s")

    # Timestamp ou datetime → on garde juste l'heure
    if isinstance(x, (pd.Timestamp, datetime.datetime)):
        return pd.Timestamp(
            year=1900, month=1, day=1,
            hour=x.hour, minute=x.minute, second=x.second
        )

    # Texte
    try:
        t = pd.to_datetime(str(x).strip(), errors="coerce")
        if pd.isna(t):
            return pd.NaT
        return pd.Timestamp(
            year=1900, month=1, day=1,
            hour=t.hour, minute=t.minute, second=t.second
        )
    except Exception:
        return pd.NaT


In [45]:
def get_durees_blocs(pid, feuille):
    """
    Récupère les durées pour les 3 blocs et la vidéo pour un PID donné et une feuille donnée.
    Applique un correctif spécifique pour 0101EMS.
    """
    pid = str(pid).upper()

    # ===== Bloc 1 =====
    df_b1 = pd.DataFrame(resultats_bloc1[feuille]["df"])
    ligne = df_b1.loc[df_b1["Numero_inclusion"].str.upper() == pid]

    if ligne.empty:
        raise ValueError(f"PID {pid} absent du bloc 1 dans la feuille {feuille}")

    d1 = ligne["duree_bloc1"].values[0]
    condition = ligne["Condition"].values[0]

    # ===== Durée vidéo =====
    t_fin_anamnese = convertir_et_normaliser_heure(
        ligne["heure_fin_anamnese_v2"].values[0]
    )
    t_debut_rlri = convertir_et_normaliser_heure(
        ligne["heure_rlri16imm_debut_V2"].values[0]
    )

    if pd.notna(t_fin_anamnese) and pd.notna(t_debut_rlri):
        delta = (t_debut_rlri - t_fin_anamnese) / pd.Timedelta(seconds=1)  # en secondes
        if delta < 0:
            delta += 24*3600
    else:
        delta = 0
    duree_video_sec = delta

    # ===== DEBUG =====
    print("\n=== DEBUG VIDEO ===")
    print("Feuille :", feuille)
    print("PID :", pid)
    print("heure_fin_anamnese brute :", ligne["heure_fin_anamnese_v2"].values[0])
    print("heure_rlri16imm_debut brute :", ligne["heure_rlri16imm_debut_V2"].values[0])
    print("t_fin_anamnese :", t_fin_anamnese)
    print("t_debut_rlri  :", t_debut_rlri)
    print("delta final (s) :", delta)
    print("===================")

    # ===== Bloc 2 =====
    if condition == "Standard":
        df_b2 = pd.DataFrame(resultats_bloc2_S[feuille]["df"])
        duree_col = "duree_bloc2S"
    else:
        df_b2 = pd.DataFrame(resultats_bloc2_R[feuille]["df"])
        duree_col = "duree_bloc2R"

    # ⚠️ Correction spécifique pour 0101EMS
    if pid == "0101EMS":
        for col in ["heure_rlri16imm_debut_V2", "heure_nback_v2 (consigne)"]:
            val = df_b2.loc[df_b2["Numero_inclusion"].str.upper() == pid, col].values[0]
            ts = convertir_et_normaliser_heure(val)
            df_b2.loc[df_b2["Numero_inclusion"].str.upper() == pid, col] = ts

        # Recalculer la durée pour ce PID
        start = df_b2.loc[df_b2["Numero_inclusion"].str.upper() == pid, "heure_rlri16imm_debut_V2"].values[0]
        end   = df_b2.loc[df_b2["Numero_inclusion"].str.upper() == pid, "heure_nback_v2 (consigne)"].values[0]
        df_b2.loc[df_b2["Numero_inclusion"].str.upper() == pid, duree_col] = (end - start) / pd.Timedelta(minutes=1)

    # ===== Récupérer d2 normalement =====
    d2 = df_b2.loc[df_b2["Numero_inclusion"].str.upper() == pid, duree_col].values[0]

    # ⚠️ Ajouter delta vidéo uniquement pour 0101EMS
    if pid == "0101EMS":
        d2 += duree_video_sec / 60  # secondes → minutes

    # ===== Bloc 3 =====
    df_b3 = pd.DataFrame(resultats_bloc3[feuille]["df"])
    d3 = df_b3.loc[df_b3["Numero_inclusion"].str.upper() == pid, "duree_bloc3"].values[0]

    return {
        "bloc1_sec": d1 * 60,
        "video_sec": duree_video_sec,
        "bloc2_sec": d2 * 60,
        "bloc3_sec": d3 * 60,
        "condition": condition
    }

In [51]:
# ===== Dossiers =====
input_dir = os.path.join(os.path.expanduser("~"), "Desktop", "EDA_v2_tronqués")
output_dir = os.path.join(os.path.expanduser("~"), "Desktop", "EDA_v2_tronque_blocs")
os.makedirs(output_dir, exist_ok=True)

resultats_decoupage = {}

# ===== Boucle UNIQUEMENT sur les fichiers tronqués =====
for fichier in os.listdir(input_dir):

    if not fichier.lower().endswith(".csv"):
        continue

    chemin_eda = os.path.join(input_dir, fichier)

    # ===== Extraction PID depuis le nom =====
    match = re.search(r"\d{4}[A-Z]{3}", fichier.upper())
    if not match:
        print(f"⚠️ PID introuvable dans {fichier}")
        continue

    pid = match.group()

    # ===== Récupération de la feuille associée =====
    ligne = df_global.loc[df_global["Numero_inclusion"] == pid]
    if ligne.empty:
        print(f"⚠️ Feuille introuvable pour {pid}")
        continue

    feuille = ligne["Feuille"].values[0]

    print(f"\n--- Découpage {pid} ({feuille}) ---")

    # ===== Lecture EDA =====
    try:
        df_rr = pd.read_csv(chemin_eda,header=None)
    except Exception as e:
        print(f"❌ Lecture impossible {pid} : {e}")
        continue

    # ===== Récupération des durées =====
    try:
        durees = get_durees_blocs(pid, feuille)
    except Exception as e:
        print(f"⚠️ Impossible de récupérer les durées pour {pid} ({feuille}) : {e}")
        continue
        
    delta_row = df_offset_eda.loc[df_offset_eda["Numero_inclusion"].str.upper() == pid.upper()]
    if delta_row.empty:
        delta_val = 0
    else:
        delta_val = float(delta_row["delta_retard_sec"].values[0])
        
    # ===== Découpage =====
    decoupe = decouper_eda_par_bloc(
        df_eda=df_rr,
        delta = delta_val,
        duree_bloc1=durees["bloc1_sec"],
        duree_video=durees["video_sec"],
        duree_bloc2=durees["bloc2_sec"],
        duree_bloc3=durees["bloc3_sec"]
    )

    if decoupe is None:
        print(f"⚠️ Découpage vide pour {pid}")
        continue

    # ===== Sauvegarde par bloc =====
    for bloc in ["bloc1", "bloc2", "bloc3"]:
        df_bloc = decoupe[bloc]

        if df_bloc.empty:
            print(f"⚠️ {pid} {bloc} vide")
            continue

        nom_sortie = f"{pid}_{bloc}.csv"
        df_bloc.to_csv(
            os.path.join(output_dir, nom_sortie),
            index=False
        )

    # ===== Stockage mémoire =====
    resultats_decoupage[pid] = decoupe

    print(
        f"✔ {pid} | "
        f"B1={len(decoupe['bloc1'])} | "
        f"B2={len(decoupe['bloc2'])} | "
        f"B3={len(decoupe['bloc3'])}"
    )

print("\n✅ Découpage terminé")
print("📁 Résultats dans :", output_dir)


--- Découpage 0101CAR (APHM) ---

=== DEBUG VIDEO ===
Feuille : APHM
PID : 0101CAR
heure_fin_anamnese brute : 08:50:00
heure_rlri16imm_debut brute : 08:57:00
t_fin_anamnese : 1900-01-01 08:50:00
t_debut_rlri  : 1900-01-01 08:57:00
delta final (s) : 420.0
0    1.530858e+09
Name: 0, dtype: float64
0    4.0
Name: 1, dtype: float64
✔ 0101CAR | B1=6000 | B2=15360 | B3=14401

--- Découpage 0101EMS (LPC) ---

=== DEBUG VIDEO ===
Feuille : LPC
PID : 0101EMS
heure_fin_anamnese brute : 10:33:00
heure_rlri16imm_debut brute :  10:35
t_fin_anamnese : 1900-01-01 10:33:00
t_debut_rlri  : 1900-01-01 10:35:00
delta final (s) : 120.0
0    1.548148e+09
Name: 0, dtype: float64
0    4.0
Name: 1, dtype: float64
✔ 0101EMS | B1=5360 | B2=9360 | B3=9601

--- Découpage 0102PCR (APHM) ---

=== DEBUG VIDEO ===
Feuille : APHM
PID : 0102PCR
heure_fin_anamnese brute : 09:24:00
heure_rlri16imm_debut brute : 09:30:00
t_fin_anamnese : 1900-01-01 09:24:00
t_debut_rlri  : 1900-01-01 09:30:00
delta final (s) : 360.0
0   

In [ ]:
## Traitment pour passer le fichier dans kubios